# Quy trình chạy mô hình ngập lụt đường bộ TP.HCM trên Google Colab (100% Google Drive — Không dùng Git/GitHub)

Notebook này hoạt động **độc lập 100% trên Google Drive & Colab**:
- **Không dùng Git / GitHub / `uv`** (sử dụng trực tiếp Python sẵn có của Google Colab)
- **Tối ưu phần cứng cho Colab A100** (12 vCPUs, 80GB+ RAM, tải dữ liệu song song đa luồng trực tiếp vào ổ SSD NVMe `/content/raw_cache`)

**Cách chạy:** Chọn `Runtime` -> `Run all` (Chạy tất cả).


In [ ]:
import hashlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, datetime
from pathlib import Path

# ==================== CẤU HÌNH THAM SỐ ====================
DRIVE_ROOT = Path("/content/drive/MyDrive/hcmc-road-flood-ml")
RUN_ID = ""  # Để trống sẽ tự động tạo mã lần chạy mới (VD: hcmc_colab_20261106_111000)
PUBLISH_INTERIM = False  # Đặt True nếu muốn lưu cả dữ liệu trung gian (interim) về Drive
OUTPUT_RESERVE_GIB = 20  # Dung lượng trống dự phòng trên Colab (GiB)
CHECKOUT = Path("/content/hcmc-road-flood-ml")
LOCAL_RAW_CACHE = Path("/content/raw_cache")
# ==========================================================

# 1. Tự động kết nối (Mount) Google Drive nếu chưa gắn
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")

if not DRIVE_ROOT.is_dir():
    alt_drive = Path("/content/drive/MyDrive/datasetHackathon/hcmc-road-flood-ml")
    if alt_drive.is_dir():
        DRIVE_ROOT = alt_drive
    else:
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = DRIVE_ROOT.resolve()

# 2. Tự động tạo RUN_ID mới (hoặc tiếp tục RUN_ID đang chạy dở trên /content/hcmc-road-flood-ml)
if not RUN_ID:
    _incomplete_runs = sorted(
        [
            d
            for d in (CHECKOUT / "data" / "processed").glob("hcmc_colab_*")
            if d.is_dir()
            and (d / "road_features.parquet").is_file()
            and not (DRIVE_ROOT / "runs" / d.name / "completion_manifest.json").is_file()
        ],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if _incomplete_runs:
        RUN_ID = _incomplete_runs[0].name
        print(f"Phát hiện phiên chạy đang dở trên SSD Colab, tự động tiếp tục: {RUN_ID}")
    else:
        RUN_ID = f"hcmc_colab_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", RUN_ID) or RUN_ID in {".", ".."}:
    raise ValueError("RUN_ID không hợp lệ (chỉ dùng chữ cái, số, dấu _, ., -).")
if (DRIVE_ROOT / "runs" / RUN_ID / "completion_manifest.json").is_file():
    raise FileExistsError(
        f"RUN_ID '{RUN_ID}' đã hoàn tất trên Drive. Hãy đổi tên khác hoặc để trống RUN_ID."
    )

# 3. Tự động tìm thư mục chứa dữ liệu gốc (raw/ hoặc data/raw/)
RAW_CANDIDATES = [
    DRIVE_ROOT / "raw",
    DRIVE_ROOT / "data" / "raw",
    DRIVE_ROOT / "hcmc-road-flood-ml" / "raw",
    DRIVE_ROOT / "hcmc-road-flood-ml" / "data" / "raw",
]
RAW_ROOT = next(
    (
        p.resolve()
        for p in RAW_CANDIDATES
        if (p / "HCMC_Floods_BDD.gpkg").is_file() or (p / "dataverse_files").is_dir()
    ),
    None,
)
if RAW_ROOT is None:
    RAW_ROOT = next((p.resolve() for p in RAW_CANDIDATES if p.is_dir()), None)
if RAW_ROOT is None:
    (DRIVE_ROOT / "raw").mkdir(parents=True, exist_ok=True)
    RAW_ROOT = (DRIVE_ROOT / "raw").resolve()

STARTED_AT = datetime.now(UTC).isoformat()
print("Đã kết nối Google Drive thành công!")
print(f"- Thư mục Drive : {DRIVE_ROOT}")
print(f"- Thư mục Raw   : {RAW_ROOT}")
print(f"- Mã lần chạy   : {RUN_ID}")

## 1. Sao chép mã nguồn & Cài đặt nhanh thư viện còn thiếu
Chỉ cài đặt đúng các thư viện chưa có sẵn trên Colab (`osmium`, `rasterio`, `pyogrio`), cấu hình đa nhân cho A100 và hỗ trợ in log trực tiếp (real-time streaming).


In [ ]:
import ast
import importlib
import tomllib

# 1. Sao chép gọn các file mã nguồn cần thiết vào ổ SSD nội bộ (/content/hcmc-road-flood-ml), giữ nguyên cache data/reports đang chạy dở
CHECKOUT.mkdir(parents=True, exist_ok=True)

REQUIRED_ITEMS = ("pyproject.toml", "uv.lock", ".python-version", "configs", "src", "scripts", "tests")
for _old_item in REQUIRED_ITEMS:
    _old_p = CHECKOUT / _old_item
    if _old_p.is_dir():
        shutil.rmtree(_old_p)
    elif _old_p.is_file():
        _old_p.unlink()


def _is_valid_source_dir(cand: Path) -> bool:
    if not cand.is_dir():
        return False
    for name in (*REQUIRED_ITEMS, "src/flood_ml/inference.py", "scripts/audit_ranking_spatial_bias.py", "scripts/score_all_roads.py"):
        if not (cand / name).exists():
            return False
    if "fit_local_basin_stats" not in (cand / "src/flood_ml/inference.py").read_text(encoding="utf-8", errors="ignore"):
        return False
    if not ((cand / "data" / "raw" / "manifest.yaml").is_file() or (RAW_ROOT / "manifest.yaml").is_file()):
        return False
    try:
        tomllib.loads((cand / "pyproject.toml").read_text(encoding="utf-8"))
        for sub in ("src", "scripts", "tests"):
            for py_f in (cand / sub).rglob("*.py"):
                ast.parse(py_f.read_text(encoding="utf-8"), filename=str(py_f))
    except Exception:
        return False
    return True


SOURCE_CANDIDATES = (
    DRIVE_ROOT,
    DRIVE_ROOT / "hcmc-road-flood-ml",
    DRIVE_ROOT.parent / "hcmc-road-flood-ml",
)
SOURCE_DIR = next((c for c in SOURCE_CANDIDATES if _is_valid_source_dir(c)), None)

if SOURCE_DIR is None:
    print(f"Chưa có đủ mã nguồn trong {DRIVE_ROOT}; đang tự động tải và đồng bộ mã nguồn chuẩn về Drive...")
    with tempfile.TemporaryDirectory(dir="/content") as _tmp_src:
        _extracted_root = None
        try:
            _zip_file = Path(_tmp_src) / "repo.zip"
            _req = urllib.request.Request(
                "https://github.com/duygri/hcmc-road-flood-ml/archive/refs/heads/main.zip",
                headers={"User-Agent": "Mozilla/5.0"},
            )
            with urllib.request.urlopen(_req) as _resp, _zip_file.open("wb") as _out:
                shutil.copyfileobj(_resp, _out)
            with zipfile.ZipFile(_zip_file, "r") as _zf:
                _zf.extractall(_tmp_src)
            _extracted_root = next(
                (p for p in Path(_tmp_src).iterdir() if p.is_dir() and (p / "pyproject.toml").is_file()),
                None,
            )
        except Exception:
            _clone_dir = Path(_tmp_src) / "repo_clone"
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/duygri/hcmc-road-flood-ml.git", str(_clone_dir)],
                check=True,
            )
            _extracted_root = _clone_dir
        if _extracted_root is None:
            raise FileNotFoundError(f"Không tìm thấy pyproject.toml trong {DRIVE_ROOT}.")
        for _item in REQUIRED_ITEMS:
            _s = _extracted_root / _item
            _d = DRIVE_ROOT / _item
            if _s.is_dir():
                if _d.exists():
                    shutil.rmtree(_d)
                shutil.copytree(_s, _d, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "local"))
            elif _s.is_file():
                shutil.copy2(_s, _d)
        _mf_src = _extracted_root / "data" / "raw" / "manifest.yaml"
        if _mf_src.is_file():
            (DRIVE_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
            shutil.copy2(_mf_src, DRIVE_ROOT / "data" / "raw" / "manifest.yaml")
            if not (RAW_ROOT / "manifest.yaml").is_file():
                shutil.copy2(_mf_src, RAW_ROOT / "manifest.yaml")
    SOURCE_DIR = DRIVE_ROOT
    print(f"Đã đồng bộ đầy đủ mã nguồn vào Google Drive: {DRIVE_ROOT}")

for item_name in REQUIRED_ITEMS:
    src_path = SOURCE_DIR / item_name
    dst_path = CHECKOUT / item_name
    if src_path.is_dir():
        shutil.copytree(
            src_path,
            dst_path,
            ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "local"),
        )
    elif src_path.is_file():
        shutil.copy2(src_path, dst_path)

(CHECKOUT / "data" / "raw").mkdir(parents=True, exist_ok=True)
for manifest_candidate in (SOURCE_DIR / "data" / "raw" / "manifest.yaml", RAW_ROOT / "manifest.yaml"):
    if manifest_candidate.is_file():
        shutil.copy2(manifest_candidate, CHECKOUT / "data" / "raw" / "manifest.yaml")
        break

# Đảm bảo run.py hoạt động 100% không cần Git và raster.py được tối ưu tốc độ trên Colab
run_py_path = CHECKOUT / "src" / "flood_ml" / "utils" / "run.py"
if run_py_path.is_file():
    run_py_path.write_text(
        run_py_path.read_text(encoding="utf-8").replace(
            'return {"commit": None, "dirty": None}',
            'return {"commit": "local_drive", "dirty": False}',
        ),
        encoding="utf-8",
    )
raster_py_path = CHECKOUT / "src" / "flood_ml" / "features" / "raster.py"
if raster_py_path.is_file():
    r_txt = raster_py_path.read_text(encoding="utf-8")
    r_txt = r_txt.replace("GDAL_CACHEMAX=64 * 1024 * 1024", "GDAL_CACHEMAX=512 * 1024 * 1024")
    if "slope_tile = _slope(dem_tile)" not in r_txt:
        r_txt = r_txt.replace(
            "def _sample(values, transform, geometry):\n    selected = geometry_mask(\n        [geometry],\n        out_shape=values.shape,\n        transform=transform,\n        all_touched=False,\n        invert=True,\n    )",
            "def _sample(values, transform, geometry, mask=None):\n    selected = mask if mask is not None else geometry_mask([geometry], out_shape=values.shape, transform=transform, all_touched=False, invert=True)",
        )
        r_txt = r_txt.replace(
            "                lc_tile, lc_transform = _warp_window(",
            "                slope_tile = _slope(dem_tile)\n                lc_tile, lc_transform = _warp_window(",
        )
        old_loop = (
            "                    dem_bounds = (\n"
            "                        local.union(neighborhood)\n"
            "                        .buffer(math.sqrt(2) * DEM_RESOLUTION)\n"
            "                        .bounds\n"
            "                    )\n"
            "                    dem, dt = _window_from_tile(\n"
            "                        dem_tile, dem_transform, dem_bounds, DEM_RESOLUTION\n"
            "                    )\n"
            "                    heights, local_fraction = _sample(dem, dt, local)\n"
            "                    nearby, nearby_fraction = _sample(dem, dt, neighborhood)\n"
            "                    slopes, slope_fraction = _sample(_slope(dem), dt, local)"
        )
        new_loop = (
            "                    local_dem, local_dt = _window_from_tile(dem_tile, dem_transform, local.bounds, DEM_RESOLUTION)\n"
            "                    local_slope, _ = _window_from_tile(slope_tile, dem_transform, local.bounds, DEM_RESOLUTION)\n"
            "                    local_mask = geometry_mask([local], out_shape=local_dem.shape, transform=local_dt, all_touched=False, invert=True)\n"
            "                    heights, local_fraction = _sample(local_dem, local_dt, local, mask=local_mask)\n"
            "                    slopes, slope_fraction = _sample(local_slope, local_dt, local, mask=local_mask)\n"
            "                    nb_dem, nb_dt = _window_from_tile(dem_tile, dem_transform, neighborhood.bounds, DEM_RESOLUTION)\n"
            "                    nearby, nearby_fraction = _sample(nb_dem, nb_dt, neighborhood)"
        )
        r_txt = r_txt.replace(old_loop, new_loop)
    raster_py_path.write_text(r_txt, encoding="utf-8")

# 2. Chỉ cài đặt những thư viện thực sự chưa có trên Colab (tiết kiệm 80% thời gian pip)
REQUIRED_PACKAGES = {
    "osmium": "osmium>=4.0",
    "rasterio": "rasterio>=1.4",
    "pyogrio": "pyogrio>=0.10",
    "geopandas": "geopandas>=1.0",
    "pyproj": "pyproj>=3.7",
    "shapely": "shapely>=2.0",
    "pyarrow": "pyarrow>=18",
    "sklearn": "scikit-learn>=1.5",
    "joblib": "joblib>=1.5",
    "yaml": "pyyaml>=6.0",
    "tqdm": "tqdm>=4.66",
    "torch": "torch>=2.0",
    "wandb": "wandb>=0.23",
    "pytest": "pytest>=8.3",
}
missing_pkgs = [
    spec for mod, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None
]
if missing_pkgs:
    print(f"Đang cài đặt nhanh {len(missing_pkgs)} gói còn thiếu: {', '.join(missing_pkgs)} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)
    importlib.invalidate_caches()
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))

# 3. Thiết lập môi trường đa nhân cho Colab A100
CPU_CORES = str(os.cpu_count() or 4)
GPU_INFO = "CPU Runtime"
if shutil.which("nvidia-smi"):
    gpu_check = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if gpu_check.returncode == 0 and gpu_check.stdout.strip():
        GPU_INFO = gpu_check.stdout.strip()

ENV = dict(
    os.environ,
    PYTHONPATH=str(CHECKOUT / "src"),
    GDAL_CACHEMAX="1024",
    OMP_NUM_THREADS=CPU_CORES,
    OPENBLAS_NUM_THREADS=CPU_CORES,
    MKL_NUM_THREADS=CPU_CORES,
)
ENV.pop("WANDB_API_KEY", None)
ENV.setdefault("WANDB_ENTITY", "duygri-saigon-university-sgu-")
ENV.setdefault("WANDB_PROJECT", "hcmc-road-flood-ml")
try:
    from google.colab import userdata

    ENV["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY") or ""
except Exception:
    ENV.pop("WANDB_API_KEY", None)
if not ENV.get("WANDB_API_KEY"):
    ENV.pop("WANDB_API_KEY", None)

REPORTS = CHECKOUT / "reports" / "runs" / RUN_ID
REPORTS.mkdir(parents=True, exist_ok=True)


def redact_tracking_secret(text: str) -> str:
    secret = ENV.get("WANDB_API_KEY")
    return text.replace(secret, "[REDACTED]") if secret else text


def run_project(args, name):
    cmd_args = list(map(str, args))
    if cmd_args[0] == "python":
        cmd_args[0] = sys.executable
    elif cmd_args[0] == "pytest":
        cmd_args = [sys.executable, "-m", "pytest", *cmd_args[1:]]
    result = subprocess.run(
        cmd_args, check=False, cwd=CHECKOUT, env=ENV, capture_output=True, text=True
    )
    stdout = redact_tracking_secret(result.stdout)
    stderr = redact_tracking_secret(result.stderr)
    (REPORTS / f"{name}.stdout.log").write_text(stdout, encoding="utf-8")
    (REPORTS / f"{name}.stderr.log").write_text(stderr, encoding="utf-8")
    if stdout:
        print(stdout, end="" if stdout.endswith("\n") else "\n")
    if stderr:
        print(stderr, end="" if stderr.endswith("\n") else "\n")
    result.check_returncode()
    return result


COMMON = [
    "--config",
    "configs/experiments.yaml",
    "--repo-root",
    str(CHECKOUT),
    "--run-id",
    RUN_ID,
]
print(f"Sẵn sàng! Python {platform.python_version()} | {CPU_CORES} vCPUs | {GPU_INFO}")

## 2. Tải dữ liệu song song đa luồng vào ổ SSD NVMe (`/content/raw_cache`) & Đồng bộ Manifest
Tối ưu tốc độ: Tải song cả 3 nguồn dữ liệu (`OSM`, `WorldCover`, `FABDEM`) thẳng vào ổ SSD NVMe `/content/raw_cache` của máy A100, kiểm tra SHA-256 đa luồng trong 1 lượt duy nhất và lưu bản sao về Google Drive.


In [ ]:
import yaml

LOCAL_RAW_CACHE.mkdir(parents=True, exist_ok=True)
(LOCAL_RAW_CACHE / "fabdem_tiles").mkdir(parents=True, exist_ok=True)
RAW_ROOT.mkdir(parents=True, exist_ok=True)
(RAW_ROOT / "fabdem_tiles").mkdir(parents=True, exist_ok=True)

# 1. Lấy 5 file IRD từ dataverse_files hoặc RAW_ROOT trên Drive đưa thẳng vào SSD (/content/raw_cache)
IRD_FILES = (
    ("HCMC_Floods_BDD.gpkg", "HCMC_floods/Layer_files/HCMC_Floods_BDD.gpkg"),
    ("HCMC_new_limits.gpkg", "HCMC_floods/Layer_files/HCMC_new_limits.gpkg"),
    ("HCMC_water.gpkg", "HCMC_floods/Layer_files/HCMC_water.gpkg"),
    ("Metadata-form-HCMC-floods.pdf", "HCMC_floods/Metadata-form-HCMC-floods.pdf"),
    ("ReadMe_10_23708_8Y16HU.txt", "ReadMe_10_23708_8Y16HU.txt"),
)
SEARCH_DIRS = (RAW_ROOT, DRIVE_ROOT, DRIVE_ROOT / "data" / "raw", DRIVE_ROOT / "hcmc-road-flood-ml" / "data" / "raw")
for dst_name, rel_src in IRD_FILES:
    cache_dst = LOCAL_RAW_CACHE / dst_name
    drive_dst = RAW_ROOT / dst_name
    if not drive_dst.is_file():
        for sdir in SEARCH_DIRS:
            for cand in (sdir / dst_name, sdir / "dataverse_files" / rel_src, sdir / rel_src):
                if cand.is_file():
                    shutil.copy2(cand, drive_dst)
                    break
            if drive_dst.is_file():
                break
    if not cache_dst.is_file() and drive_dst.is_file():
        shutil.copy2(drive_dst, cache_dst)

missing_ird = [dst_name for dst_name, _ in IRD_FILES if not (LOCAL_RAW_CACHE / dst_name).is_file()]
if missing_ird:
    print(f"Đang tự động tải 5 file IRD DataSuds (DOI: 10.23708/8Y16HU) cho: {', '.join(missing_ird)} ...")
    with tempfile.TemporaryDirectory(dir="/content") as _ird_tmp:
        _ird_zip = Path(_ird_tmp) / "dataverse.zip"
        _req = urllib.request.Request(
            "https://dataverse.ird.fr/api/access/dataset/:persistentId/?persistentId=doi:10.23708/8Y16HU",
            headers={"User-Agent": "Mozilla/5.0"},
        )
        with urllib.request.urlopen(_req) as _resp, _ird_zip.open("wb") as _out:
            shutil.copyfileobj(_resp, _out, length=4 * 1024 * 1024)
        with zipfile.ZipFile(_ird_zip, "r") as _zf:
            _zf.extractall(_ird_tmp)
        for _nested in list(Path(_ird_tmp).rglob("*.zip")):
            if _nested != _ird_zip:
                with zipfile.ZipFile(_nested, "r") as _nzf:
                    _nzf.extractall(_ird_tmp)
        for dst_name, _ in IRD_FILES:
            found_f = next((p for p in Path(_ird_tmp).rglob(dst_name) if p.is_file()), None)
            if found_f is not None:
                shutil.copy2(found_f, LOCAL_RAW_CACHE / dst_name)
                shutil.copy2(found_f, RAW_ROOT / dst_name)


# 2. Hàm tải trực tiếp vào ổ SSD NVMe (/content/raw_cache) rồi đồng bộ về Drive
def fetch_or_stage_single(rel_path: str, url: str) -> None:
    cache_file = LOCAL_RAW_CACHE / rel_path
    drive_file = RAW_ROOT / rel_path
    if cache_file.is_file() and cache_file.stat().st_size > 0:
        return
    if drive_file.is_file() and drive_file.stat().st_size > 0:
        print(f"Đang nạp từ Drive vào SSD: {rel_path} ...")
        shutil.copy2(drive_file, cache_file)
        return
    print(f"Đang tải từ Internet vào SSD: {rel_path} ...")
    tmp = cache_file.with_suffix(cache_file.suffix + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, tmp.open("wb") as out:
        shutil.copyfileobj(resp, out, length=4 * 1024 * 1024)
    tmp.replace(cache_file)
    shutil.copy2(cache_file, drive_file)
    print(f"Hoàn tất: {rel_path} ({cache_file.stat().st_size / 1024 / 1024:.1f} MB)")


def fetch_or_stage_fabdem() -> None:
    tile_names = [f"{t}_FABDEM_V1-2.tif" for t in ("N10E106", "N10E107", "N11E106", "N11E107")]
    missing_in_cache = []
    for tname in tile_names:
        cfile = LOCAL_RAW_CACHE / "fabdem_tiles" / tname
        dfile = RAW_ROOT / "fabdem_tiles" / tname
        if not cfile.is_file():
            if dfile.is_file():
                shutil.copy2(dfile, cfile)
            else:
                missing_in_cache.append(tname)
    if not missing_in_cache:
        return
    url = "https://data.bris.ac.uk/datasets/s5hqmjcdj8yo2ibzi9b4ew3sn/N10E100-N20E110_FABDEM_V1-2.zip"
    print("Đang tải bộ địa hình FABDEM v1.2 vào SSD ...")
    with tempfile.TemporaryDirectory(dir="/content") as tmp_dir:
        zip_path = Path(tmp_dir) / "FABDEM.zip"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, zip_path.open("wb") as out:
            shutil.copyfileobj(resp, out, length=4 * 1024 * 1024)
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.infolist():
                mname = Path(member.filename).name
                if mname in missing_in_cache:
                    cfile = LOCAL_RAW_CACHE / "fabdem_tiles" / mname
                    dfile = RAW_ROOT / "fabdem_tiles" / mname
                    with zf.open(member) as src, cfile.open("wb") as out:
                        shutil.copyfileobj(src, out, length=4 * 1024 * 1024)
                    shutil.copy2(cfile, dfile)
    print("Hoàn tất giải nén 4 tile FABDEM v1.2!")


# Chạy tải song song cả 3 nguồn dữ liệu cùng lúc bằng ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [
        pool.submit(
            fetch_or_stage_single,
            "vietnam-latest.osm.pbf",
            "https://download.geofabrik.de/asia/vietnam-latest.osm.pbf",
        ),
        pool.submit(
            fetch_or_stage_single,
            "ESA_WorldCover_10m_2021_v200_N09E105_Map.tif",
            "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N09E105_Map.tif",
        ),
        pool.submit(fetch_or_stage_fabdem),
    ]
    for fut in futures:
        fut.result()

# 3. Băm SHA-256 song song đa luồng ngay trên ổ SSD NVMe & Đồng bộ manifest.yaml trong 1 lượt duy nhất
manifest_path = CHECKOUT / "data" / "raw" / "manifest.yaml"
manifest_doc = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
all_assets = [asset for ds in manifest_doc["datasets"] for asset in ds["assets"]]


def hash_asset(asset: dict) -> tuple[str, int, str]:
    fpath = (LOCAL_RAW_CACHE / asset["path"]).resolve()
    if not fpath.is_file():
        raise FileNotFoundError(f"Thiếu file dữ liệu gốc: {asset['path']}")
    with fpath.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    return asset["path"], fpath.stat().st_size, digest


with ThreadPoolExecutor(max_workers=min(11, int(CPU_CORES))) as pool:
    measured = {path: (sz, dg) for path, sz, dg in pool.map(hash_asset, all_assets)}

manifest_changed = False
for asset in all_assets:
    sz, dg = measured[asset["path"]]
    if asset.get("bytes") != sz or asset.get("sha256") != dg:
        asset["bytes"] = sz
        asset["sha256"] = dg
        manifest_changed = True

if manifest_changed:
    manifest_path.write_text(
        yaml.safe_dump(manifest_doc, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    shutil.copy2(manifest_path, RAW_ROOT / "manifest.yaml")
    print("Đã tự động đồng bộ checksum mới vào manifest.yaml!")

required_bytes = sum(sz for sz, _ in measured.values()) + int(OUTPUT_RESERVE_GIB * 1024**3)
if shutil.disk_usage("/content").free < required_bytes:
    raise RuntimeError("Không đủ dung lượng trống tại /content")

# 4. Ghi cấu hình nguồn dữ liệu trỏ thẳng vào ổ SSD NVMe (/content/raw_cache)
LOCAL = CHECKOUT / "configs" / "local"
LOCAL.mkdir(parents=True, exist_ok=True)
(LOCAL / "sources.yaml").write_text(
    json.dumps({"sources": {"raw_root": str(LOCAL_RAW_CACHE)}}), encoding="utf-8"
)
(REPORTS / "preflight.stdout.log").write_text(
    json.dumps({"verified_files": len(measured), "required_scratch_bytes": required_bytes}) + "\n",
    encoding="utf-8",
)

REVIEW_ARGS = []
review_hashes = {}
for filename, flag in (
    ("review_decisions.csv", "--match-reviews"),
    ("event_reviews.csv", "--event-reviews"),
):
    review_candidates = [
        DRIVE_ROOT / "review" / filename,
        DRIVE_ROOT / "configs" / "local" / filename,
    ]
    review = next((p for p in review_candidates if p.is_file()), None)
    if review is not None:
        destination = LOCAL / filename
        shutil.copyfile(review, destination)
        with destination.open("rb") as stream:
            review_hashes[filename] = hashlib.file_digest(stream, "sha256").hexdigest()
        REVIEW_ARGS.extend([flag, f"configs/local/{filename}"])
        print(f"Đã nạp file review: {filename}")
    else:
        print(f"Không có {filename}; giữ nguyên trạng thái chưa review (pending_positive_review).")

print(f"Hoàn tất chuẩn bị & xác thực {len(measured)}/11 file dữ liệu gốc trên SSD NVMe!")

## 3. Chạy kiểm thử nhanh (Smoke Test)
Kiểm tra toàn bộ luồng xử lý bằng dữ liệu giả lập trước khi chạy trên dữ liệu thật.


In [ ]:
run_project(["pytest", "tests/integration/test_track_b.py", "-q"], "synthetic_smoke")
print("Smoke test thành công!")

## 4. Chạy Pipeline xử lý dữ liệu & Baseline MCDA (Theo dõi tiến độ bằng `tqdm`)
Trích xuất mạng lưới đường, đặc trưng địa hình/thủy văn/bề mặt, gán nhãn, chia phân vùng không gian và tính điểm nhạy cảm ngập lụt cơ sở (MCDA) với thanh tiến trình `tqdm`.


In [ ]:
import hashlib
import json
import shutil
from pathlib import Path
from tqdm.auto import tqdm

LOCAL = CHECKOUT / "configs" / "local"
if "review_hashes" not in globals():
    review_hashes = {}


def reconcile_reviews_and_sync_stage_cache() -> None:
    roads_p = CHECKOUT / "data" / "interim" / RUN_ID / "road_segments.gpkg"
    obs_p = CHECKOUT / "data" / "interim" / RUN_ID / "flood_observations.gpkg"
    mr_p = LOCAL / "review_decisions.csv"
    er_p = LOCAL / "event_reviews.csv"
    if roads_p.is_file() and obs_p.is_file() and mr_p.is_file() and er_p.is_file():
        import geopandas as gpd
        import pandas as pd
        from flood_ml.config import PipelineError, load_settings
        from flood_ml.labeling.matching import match_candidates
        from flood_ml.labeling.reviews import apply_reviews

        cfg_settings = load_settings(CHECKOUT / "configs" / "experiments.yaml", repo_root=CHECKOUT)
        roads_gdf = gpd.read_file(roads_p)
        obs_gdf = gpd.read_file(obs_p)
        cands_df = match_candidates(obs_gdf, roads_gdf, cfg_settings.labeling)
        mr_df = pd.read_csv(mr_p, dtype=str, keep_default_na=False)
        er_df = pd.read_csv(er_p, dtype=str, keep_default_na=False)
        needs_reconcile = False
        try:
            apply_reviews(roads_gdf, obs_gdf, cands_df, mr_df, er_df)
        except PipelineError:
            needs_reconcile = True
        if needs_reconcile:
            valid_pairs = (
                cands_df.dropna(subset=["road_segment_id"])
                .drop_duplicates(["observation_id", "road_segment_id"])
                .sort_values(["observation_id", "match_distance_m", "road_segment_id"], kind="stable")
            )
            valid_pair_set = set(zip(valid_pairs["observation_id"], valid_pairs["road_segment_id"]))
            valid_obs_in_aoi = set(
                obs_gdf.loc[~obs_gdf["outside_aoi"].fillna(True), "observation_id"]
            )
            all_obs_ids = set(obs_gdf["observation_id"])

            er_clean = er_df[er_df["observation_id"].isin(all_obs_ids)].drop_duplicates(["observation_id"]).copy()
            er_obs_set = set(er_clean["observation_id"])

            acc_rows = []
            for _, r in mr_df[mr_df["decision"] == "accepted"].iterrows():
                oid = r["observation_id"].strip()
                sid = r["road_segment_id"].strip()
                if oid not in valid_obs_in_aoi or oid not in er_obs_set:
                    continue
                if (oid, sid) not in valid_pair_set:
                    obs_cands = valid_pairs[valid_pairs["observation_id"] == oid]
                    if len(obs_cands) == 0:
                        continue
                    r = r.copy()
                    r["road_segment_id"] = str(obs_cands.iloc[0]["road_segment_id"])
                acc_rows.append(r)
            acc_df = (
                pd.DataFrame(acc_rows, columns=mr_df.columns).drop_duplicates(["observation_id", "road_segment_id"])
                if acc_rows
                else pd.DataFrame(columns=mr_df.columns)
            )
            acc_pair_set = set(zip(acc_df["observation_id"], acc_df["road_segment_id"]))
            acc_obs_set = set(acc_df["observation_id"])
            er_clean = er_clean[er_clean["observation_id"].isin(acc_obs_set)].copy()

            other_df = mr_df[mr_df["decision"] != "accepted"].copy()
            other_mask = [
                (row["observation_id"].strip() in valid_obs_in_aoi)
                and ((row["observation_id"].strip(), row["road_segment_id"].strip()) in valid_pair_set)
                and ((row["observation_id"].strip(), row["road_segment_id"].strip()) not in acc_pair_set)
                for _, row in other_df.iterrows()
            ]
            other_df = other_df.loc[other_mask].drop_duplicates(["observation_id", "road_segment_id"])
            mr_clean = pd.concat([acc_df, other_df], ignore_index=True)
            apply_reviews(roads_gdf, obs_gdf, cands_df, mr_clean, er_clean)
            mr_clean.to_csv(mr_p, index=False, encoding="utf-8")
            er_clean.to_csv(er_p, index=False, encoding="utf-8")
            for fname, fpath in (("review_decisions.csv", mr_p), ("event_reviews.csv", er_p)):
                with fpath.open("rb") as strm:
                    review_hashes[fname] = hashlib.file_digest(strm, "sha256").hexdigest()
                if (DRIVE_ROOT / "review").is_dir():
                    shutil.copy2(fpath, DRIVE_ROOT / "review" / fname)
            print(f"Đã tự động đồng bộ {len(acc_df)} điểm ngập dương (accepted) khớp 100% với bản đồ OSM hiện tại!")

    run_json_f = REPORTS / "run.json"
    if run_json_f.is_file() or (REPORTS / "stages").is_dir():
        from dataclasses import replace as dc_replace
        from flood_ml.config import load_settings
        from flood_ml.pipeline import STAGE_DEPENDENCIES, STAGE_ORDER
        from flood_ml.utils.run import _hash_file, _hash_json, start_run, write_json_atomic

        cfg = load_settings(CHECKOUT / "configs" / "experiments.yaml", repo_root=CHECKOUT)
        lbl = dict(cfg.labeling)
        if mr_p.is_file():
            lbl["match_reviews"] = mr_p.relative_to(CHECKOUT).as_posix()
        if er_p.is_file():
            lbl["event_reviews"] = er_p.relative_to(CHECKOUT).as_posix()
        cfg = dc_replace(cfg, labeling=lbl, experiments={**cfg.experiments, "run_id": RUN_ID})
        run_json_f.unlink(missing_ok=True)
        (REPORTS / ".start.lock").unlink(missing_ok=True)
        (REPORTS / ".pipeline.lock").unlink(missing_ok=True)
        start_run(cfg, RUN_ID)
        new_fp = json.loads(run_json_f.read_text(encoding="utf-8"))["fingerprint"]
        synced_stages = {}
        for st in STAGE_ORDER:
            st_marker = REPORTS / "stages" / f"{st}.json"
            if not st_marker.is_file():
                continue
            st_rec = json.loads(st_marker.read_text(encoding="utf-8"))
            deps = STAGE_DEPENDENCIES[st]
            if not all(d in synced_stages for d in deps):
                continue
            outs = st_rec.get("outputs", {})
            if not outs or not all(Path(p).is_file() and _hash_file(Path(p)) == dg for p, dg in outs.items()):
                continue
            dep_outs = {d: synced_stages[d]["outputs"] for d in deps}
            st_rec["input_fingerprint"] = _hash_json(
                {"run_fingerprint": new_fp, "stage": st, "dependencies": dep_outs}
            )
            write_json_atomic(st_marker, st_rec)
            synced_stages[st] = st_rec


reconcile_reviews_and_sync_stage_cache()

PIPELINE_STAGES = (
    ("validate", "Kiểm tra nguồn dữ liệu"),
    ("aoi", "Trích xuất ranh giới TP.HCM (AOI)"),
    ("roads", "Phân đoạn mạng lưới đường OSM"),
    ("observations", "Chuẩn hóa điểm ngập lịch sử"),
    ("raster", "Trích xuất địa hình DEM & độ phủ WorldCover"),
    ("features", "Tính khoảng cách sông/kênh & tổng hợp đặc trưng"),
    ("labels", "Đối sánh điểm ngập với đoạn đường"),
    ("split", "Phân vùng không gian (Reference / Holdout / Guard)"),
    ("mcda", "Chấm điểm nhạy cảm ngập lụt MCDA"),
    ("evaluation", "Kiểm tra cổng đánh giá (Evaluation Gate)"),
    ("report", "Tổng hợp báo cáo & xuất bản đồ"),
)

pbar_stages = tqdm(PIPELINE_STAGES, desc="Tiến độ Pipeline", unit="stage")
for stage_code, stage_desc in pbar_stages:
    pbar_stages.set_postfix_str(stage_desc)
    if stage_code == "labels":
        reconcile_reviews_and_sync_stage_cache()
    stage_flag = [] if stage_code == "report" else ["--stage", stage_code]
    run_project(
        ["python", "scripts/run_pipeline.py", *COMMON, *REVIEW_ARGS, *stage_flag],
        "pipeline" if stage_code == "report" else f"stage_{stage_code}",
    )

## 5. Chạy cổng kiểm tra Track B chuẩn & Đánh giá (Evaluation)
Chạy bước kiểm tra cổng dữ liệu Track B và đánh giá cơ sở theo quy chuẩn của pipeline.


In [ ]:
run_project(
    [
        "python",
        "scripts/train.py",
        *COMMON,
        "--features",
        f"data/processed/{RUN_ID}/road_features.parquet",
        "--labels",
        f"data/processed/{RUN_ID}/road_labels.parquet",
        "--confirmed-event-segments",
        f"data/processed/{RUN_ID}/confirmed_event_segments.parquet",
        "--partitions",
        f"data/processed/{RUN_ID}/road_partitions.parquet",
    ],
    "train",
)

track_b = json.loads((REPORTS / "track_b.json").read_text(encoding="utf-8"))
print("Trạng thái cổng Track B:", track_b["status"])

evaluation_args = [
    "python",
    "scripts/evaluate.py",
    *COMMON,
    "--scores",
    f"data/processed/{RUN_ID}/road_susceptibility.parquet",
    "--roads",
    f"data/interim/{RUN_ID}/road_segments.gpkg",
    "--confirmed-event-segments",
    f"data/processed/{RUN_ID}/confirmed_event_segments.parquet",
    "--partitions",
    f"data/processed/{RUN_ID}/road_partitions.parquet",
]
if track_b["status"] == "ready":
    evaluation_args.extend(
        ["--track-b-scores", f"data/processed/{RUN_ID}/track_b_scores.parquet"]
    )
run_project(evaluation_args, "evaluate")

## 6. Đóng gói kết quả Pipeline & Lưu về Google Drive
Kiểm tra toàn vẹn mã băm của tất cả các file kết quả, tạo báo cáo tổng hợp và lưu vào `MyDrive/hcmc-road-flood-ml/runs/<RUN_ID>/`.


In [ ]:
runtime = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "uv_version": "pip-colab",
    "source_commit": "local_drive",
    "run_id": RUN_ID,
    "started_at": STARTED_AT,
    "finished_at": datetime.now(UTC).isoformat(),
    "review_hashes": review_hashes,
    "output_reserve_gib": OUTPUT_RESERVE_GIB,
}
(REPORTS / "runtime.json").write_text(
    json.dumps(runtime, indent=2) + "\n", encoding="utf-8"
)
evaluation = json.loads((REPORTS / "evaluation.json").read_text(encoding="utf-8"))
summary = (
    f"# Tổng hợp lần chạy Colab: {RUN_ID}\n\n"
    f"- **Trạng thái Track B**: `{track_b['status']}`\n"
    f"- **Trạng thái Đánh giá (Evaluation)**: `{evaluation['status']}`\n"
    f"- **Môi trường**: Python `{runtime['python_version']}` ({GPU_INFO})\n"
    f"- **Review hashes**: `{json.dumps(review_hashes, sort_keys=True)}`\n"
)
(REPORTS / "colab_summary.md").write_text(summary, encoding="utf-8")
run_json_path = REPORTS / "run.json"
if run_json_path.is_file():
    _run_doc = json.loads(run_json_path.read_text(encoding="utf-8"))
    if not (_run_doc.get("git") or {}).get("commit"):
        _run_doc["git"] = {"commit": "local_drive", "dirty": False}
        run_json_path.write_text(json.dumps(_run_doc, indent=2) + "\n", encoding="utf-8")

publish_target = DRIVE_ROOT / "runs" / RUN_ID
if publish_target.exists() and not (publish_target / "completion_manifest.json").is_file():
    shutil.rmtree(publish_target)

publish_args = [
    sys.executable,
    "scripts/publish_run.py",
    *COMMON,
    "--drive-root",
    str(DRIVE_ROOT),
]
if PUBLISH_INTERIM:
    publish_args.append("--publish-interim")

if not (publish_target / "completion_manifest.json").is_file():
    publication = subprocess.run(
        publish_args,
        check=False,
        cwd=CHECKOUT,
        env=ENV,
        capture_output=True,
        text=True,
    )
    pub_stdout = redact_tracking_secret(publication.stdout)
    pub_stderr = redact_tracking_secret(publication.stderr)
    (REPORTS / "publish.stdout.log").write_text(pub_stdout, encoding="utf-8")
    (REPORTS / "publish.stderr.log").write_text(pub_stderr, encoding="utf-8")
    if pub_stdout:
        print(pub_stdout)
    if pub_stderr:
        print(pub_stderr)
    publication.check_returncode()
print(summary)
print("Đã lưu toàn bộ kết quả vào Google Drive tại:", DRIVE_ROOT / "runs" / RUN_ID)

## 7. Huấn luyện Mô hình trên GPU A100 theo từng Epoch (`tqdm`) & Theo dõi `Capture@K%` / `Lift@K%` theo Chiều dài Đường (Chuẩn Dữ liệu Positive-Unlabeled)
Vì dữ liệu ngập báo chí là **Positive-Unlabeled (`PU`)** (các đoạn đường chưa có báo cáo là `unknown`, không phải `verified negative`), các chỉ số nhị phân truyền thống như `Accuracy`, `Precision`, `F1-Score` hay `ROC-AUC` bị sai lệch do các điểm ngập ẩn trong tập nền.

Do đó, quá trình huấn luyện `FloodSusceptibilityNet` (thí nghiệm thăm dò tính phi tuyến — **Nonlinear Ablation Study**) sử dụng trực tiếp **`Capture@K%` và `Lift@K%` theo chiều dài đường trên toàn bộ vùng `Holdout`** ở mỗi Epoch để chọn `Best Model` và vẽ 4 biểu đồ:
1. **Train BCE Loss vs Val BCE Loss** (Kiểm soát chống Overfitting)
2. **Holdout `Capture@5%`, `Capture@10%`, `Capture@20%` theo chiều dài đường qua từng Epoch** (Thước đo chính thay cho `Accuracy`)
3. **Holdout `Lift@5%`, `Lift@10%`, `Lift@20%` qua từng Epoch** (Thước đo hiệu quả ưu tiên ngân sách thay cho `F1 / ROC-AUC`)
4. **Đường cong `Gain Curve (Cumulative Length Capture)` tại Best Epoch** đặt cạnh `Logistic Regression (L2 — 9 tham số)` và `Baseline MCDA`


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Image, Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Tự động khôi phục đường dẫn nếu Colab vừa khởi động lại phiên (Restart session)
if "DRIVE_ROOT" not in globals():
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    DRIVE_ROOT = Path("/content/drive/MyDrive/hcmc-road-flood-ml")

if "CHECKOUT" not in globals():
    CHECKOUT = Path("/content/hcmc-road-flood-ml")

if "RUN_ID" not in globals():
    run_dirs = sorted(
        [d for d in (DRIVE_ROOT / "runs").glob("*") if d.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    RUN_ID = run_dirs[0].name
    print(f"Đã tự động khôi phục RUN_ID từ Google Drive: {RUN_ID}")

if "GPU_INFO" not in globals():
    GPU_INFO = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

RUN_DRIVE_DIR = DRIVE_ROOT / "runs" / RUN_ID
RUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = CHECKOUT / "data" / "processed" / RUN_ID
if not PROCESSED_DIR.is_dir():
    PROCESSED_DIR = RUN_DRIVE_DIR / "data" / "processed" / RUN_ID
ARTIFACTS_DIR = CHECKOUT / "artifacts" / RUN_ID
if not ARTIFACTS_DIR.is_dir():
    ARTIFACTS_DIR = RUN_DRIVE_DIR / "artifacts" / RUN_ID

# 1. Đọc bảng đặc trưng, phân vùng không gian, điểm MCDA và điểm ngập lịch sử
df_feat = pd.read_parquet(PROCESSED_DIR / "road_features.parquet")
df_part = pd.read_parquet(PROCESSED_DIR / "road_partitions.parquet")
df_mcda = pd.read_parquet(PROCESSED_DIR / "road_susceptibility.parquet")
df_confirmed = pd.read_parquet(PROCESSED_DIR / "confirmed_event_segments.parquet")
matches_path = PROCESSED_DIR / "road_flood_matches.parquet"
df_matches = pd.read_parquet(matches_path) if matches_path.is_file() else pd.DataFrame()

FEATURE_COLS = [
    "elevation_median_m",
    "elevation_p10_m",
    "relative_elevation_m",
    "slope_mean_deg",
    "builtup_ratio_250m",
    "vegetation_ratio_250m",
    "dist_water_m",
    "water_area_ratio_250m",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_feat.columns]

# Xác định các đoạn đường ghi nhận ngập (Verified Positive = 1, còn lại 0 = Unlabeled Background)
if len(df_confirmed) > 0 and "road_segment_id" in df_confirmed.columns:
    pos_ids = set(df_confirmed["road_segment_id"].dropna().unique())
else:
    valid_matches = df_matches.dropna(subset=["road_segment_id"])
    if "match_distance_m" in valid_matches.columns:
        close_matches = valid_matches[valid_matches["match_distance_m"] <= 50.0]
        if len(close_matches) >= 50:
            valid_matches = close_matches
    pos_ids = set(valid_matches["road_segment_id"].unique())

data = df_feat.merge(
    df_part[["road_segment_id", "partition", "block_id"]], on="road_segment_id", how="inner"
)
if "length_m" not in data.columns:
    if "road_length_m" in data.columns:
        data["length_m"] = data["road_length_m"]
    elif "road_length_m" in df_mcda.columns:
        data = data.merge(
            df_mcda[["road_segment_id", "road_length_m"]].rename(columns={"road_length_m": "length_m"}),
            on="road_segment_id",
            how="left",
        )
    else:
        data["length_m"] = 150.0
data["length_m"] = data["length_m"].fillna(150.0).astype(np.float32)
data["label"] = data["road_segment_id"].isin(pos_ids).astype(np.float32)
valid_mask = data[FEATURE_COLS].notna().all(axis=1)
if "score_eligible" in data.columns:
    valid_mask = valid_mask & data["score_eligible"].fillna(True)
clean_data = data.loc[valid_mask].reset_index(drop=True)

# Chia tập Train (reference) và Validation (holdout), lấy mẫu Presence-Background (1:3) cho hàm Loss
rng = np.random.default_rng(42)


def build_balanced_split(df_sub: pd.DataFrame, bg_ratio: int = 3) -> pd.DataFrame:
    pos_df = df_sub[df_sub["label"] == 1.0]
    bg_df = df_sub[df_sub["label"] == 0.0]
    if len(pos_df) == 0 or len(bg_df) == 0:
        return df_sub
    n_bg = min(len(bg_df), len(pos_df) * bg_ratio)
    sampled_bg = bg_df.iloc[rng.choice(len(bg_df), size=n_bg, replace=False)]
    return pd.concat([pos_df, sampled_bg], ignore_index=True).sample(
        frac=1.0, random_state=42
    )


train_raw = clean_data[clean_data["partition"] == "reference"].copy()
val_raw = clean_data[clean_data["partition"] == "holdout"].copy()
if (train_raw["label"] == 1.0).sum() < 10 or (val_raw["label"] == 1.0).sum() < 5:
    shuffled = clean_data.sample(frac=1.0, random_state=42).reset_index(drop=True)
    cut = int(len(shuffled) * 0.8)
    train_raw, val_raw = shuffled.iloc[:cut].copy(), shuffled.iloc[cut:].copy()

# Tách nội bộ tập Reference (80% Inner-Train / 20% Inner-Val) để chọn Best Epoch mà KHÔNG rò rỉ tập Holdout
ref_pos = train_raw[train_raw["label"] == 1.0].sample(frac=1.0, random_state=42)
ref_bg = train_raw[train_raw["label"] == 0.0].sample(frac=1.0, random_state=42)
pos_cut = max(1, int(len(ref_pos) * 0.8))
bg_cut = max(1, int(len(ref_bg) * 0.8))
inner_train_raw = pd.concat([ref_pos.iloc[:pos_cut], ref_bg.iloc[:bg_cut]], ignore_index=True)
inner_val_raw = pd.concat([ref_pos.iloc[pos_cut:], ref_bg.iloc[bg_cut:]], ignore_index=True)
if (inner_val_raw["label"] == 1.0).sum() == 0:
    inner_val_raw = train_raw.copy()

train_df = build_balanced_split(inner_train_raw, bg_ratio=3)
val_df = build_balanced_split(inner_val_raw, bg_ratio=3)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS].to_numpy(dtype=np.float32))
y_train = train_df["label"].to_numpy(dtype=np.float32)
X_val = scaler.transform(val_df[FEATURE_COLS].to_numpy(dtype=np.float32))
y_val = val_df["label"].to_numpy(dtype=np.float32)

# Chuẩn bị tập Inner-Val (thuộc Reference) để chọn Best Epoch và tập Holdout độc lập để đo kiểm khách quan
X_ref_val_full = scaler.transform(inner_val_raw[FEATURE_COLS].to_numpy(dtype=np.float32))
ref_val_lengths = inner_val_raw["length_m"].to_numpy(dtype=np.float64)
ref_val_labels = inner_val_raw["label"].to_numpy(dtype=np.float64)
X_holdout_full = scaler.transform(val_raw[FEATURE_COLS].to_numpy(dtype=np.float32))
holdout_lengths = val_raw["length_m"].to_numpy(dtype=np.float64)
holdout_labels = val_raw["label"].to_numpy(dtype=np.float64)
total_holdout_len = max(float(holdout_lengths.sum()), 1e-6)
flooded_holdout_len = max(float((holdout_lengths * holdout_labels).sum()), 1e-6)


def compute_pu_length_capture_lift(
    scores_arr: np.ndarray,
    budgets=(0.05, 0.10, 0.20),
    lengths_arr: np.ndarray | None = None,
    labels_arr: np.ndarray | None = None,
) -> dict[str, float]:
    l_arr = holdout_lengths if lengths_arr is None else lengths_arr
    y_arr = holdout_labels if labels_arr is None else labels_arr
    tot_l = max(float(l_arr.sum()), 1e-6)
    fl_l = max(float((l_arr * y_arr).sum()), 1e-6)
    order_idx = np.argsort(-scores_arr, kind="stable")
    ord_len = l_arr[order_idx]
    ord_flood_len = ord_len * y_arr[order_idx]
    cum_len = np.cumsum(ord_len)
    out = {}
    for k_frac in budgets:
        target_l = k_frac * tot_l
        n_sel = min(int(np.searchsorted(cum_len, target_l, side="left")) + 1, len(ord_len))
        cap_rate = float(ord_flood_len[:n_sel].sum()) / fl_l
        act_frac = float(ord_len[:n_sel].sum()) / tot_l
        pct = int(k_frac * 100)
        out[f"capture_{pct}pct"] = cap_rate
        out[f"lift_{pct}pct"] = cap_rate / max(act_frac, 1e-6)
    return out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
EPOCHS = 30
LABEL_SMOOTHING = 0.06  # Chống Overfitting cho dữ liệu Presence-Background (nhãn nền unknown có thể chứa điểm ngập ẩn)
INPUT_NOISE_STD = 0.05  # Gaussian Noise giúp mạng không học vẹt tọa độ cục bộ
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)


class FloodSusceptibilityNet(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.35),
        )
        self.res_block = nn.Sequential(
            nn.Linear(64, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.35),
            nn.Linear(64, 64),
            nn.LayerNorm(64),
        )
        self.head = nn.Sequential(
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.SiLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training and INPUT_NOISE_STD > 0:
            x = x + torch.randn_like(x) * INPUT_NOISE_STD
        h = self.input_layer(x)
        h = h + 0.5 * self.res_block(h)
        return self.head(h).squeeze(-1)


torch.manual_seed(42)
model = FloodSusceptibilityNet(len(FEATURE_COLS)).to(device)
raw_ratio = max(1.0, float((y_train == 0).sum()) / max(1.0, float((y_train == 1).sum())))
pos_weight_val = float(np.sqrt(raw_ratio))
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], device=device)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=2e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=2e-5)

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "holdout_capture_5pct": [],
    "holdout_capture_10pct": [],
    "holdout_capture_20pct": [],
    "holdout_lift_5pct": [],
    "holdout_lift_10pct": [],
    "holdout_lift_20pct": [],
    "train_acc": [],
    "val_acc": [],
    "train_precision": [],
    "val_precision": [],
    "train_recall": [],
    "val_recall": [],
    "train_f1": [],
    "val_f1": [],
    "val_auc": [],
}

BEST_PT_PATH = RUN_DRIVE_DIR / "best_model_a100.pt"
best_composite_score = -1e9
best_val_f1 = -1.0
best_epoch = 1
best_holdout_probs = None

print(
    f"Bắt đầu huấn luyện trên {device} ({GPU_INFO}) | "
    f"Train: {len(X_train):,} mẫu | Holdout toàn vùng: {len(val_raw):,} đoạn đường | {EPOCHS} Epochs"
)

X_val_t = torch.from_numpy(X_val).to(device)
y_val_t = torch.from_numpy(y_val).to(device)
y_val_smooth_t = y_val_t * (1.0 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING
X_ref_val_full_t = torch.from_numpy(X_ref_val_full).to(device)
X_holdout_full_t = torch.from_numpy(X_holdout_full).to(device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    train_preds_list, train_targets_list = [], []

    batch_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch:02d}/{EPOCHS}",
        unit="batch",
        leave=True,
    )
    for bx, by in batch_bar:
        bx, by = bx.to(device), by.to(device)
        by_smooth = by * (1.0 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by_smooth)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += float(loss.item()) * len(by)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(np.int32)
        train_preds_list.append(preds)
        train_targets_list.append(by.detach().cpu().numpy().astype(np.int32))
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()
    tr_loss = running_loss / len(X_train)
    tr_preds = np.concatenate(train_preds_list)
    tr_targets = np.concatenate(train_targets_list)
    tr_acc = float(accuracy_score(tr_targets, tr_preds))
    tr_prec = float(precision_score(tr_targets, tr_preds, zero_division=0))
    tr_rec = float(recall_score(tr_targets, tr_preds, zero_division=0))
    tr_f1 = float(f1_score(tr_targets, tr_preds, zero_division=0))

    # Đánh giá trên tập Inner-Validation (thuộc Reference) để chọn Best Epoch & theo dõi độc lập trên Holdout
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        v_loss = float(criterion(val_logits, y_val_smooth_t).item())
        v_probs = torch.sigmoid(val_logits).cpu().numpy()
        v_preds = (v_probs >= 0.5).astype(np.int32)
        v_targets = y_val.astype(np.int32)
        ref_val_probs_ep = torch.sigmoid(model(X_ref_val_full_t)).cpu().numpy()
        holdout_probs_ep = torch.sigmoid(model(X_holdout_full_t)).cpu().numpy()

    ref_val_pu = compute_pu_length_capture_lift(
        ref_val_probs_ep, lengths_arr=ref_val_lengths, labels_arr=ref_val_labels
    )
    val_cap10, val_cap20 = ref_val_pu["capture_10pct"], ref_val_pu["capture_20pct"]
    pu_metrics = compute_pu_length_capture_lift(holdout_probs_ep)
    cap5, cap10, cap20 = pu_metrics["capture_5pct"], pu_metrics["capture_10pct"], pu_metrics["capture_20pct"]
    lift5, lift10, lift20 = pu_metrics["lift_5pct"], pu_metrics["lift_10pct"], pu_metrics["lift_20pct"]

    v_acc = float(accuracy_score(v_targets, v_preds))
    v_prec = float(precision_score(v_targets, v_preds, zero_division=0))
    v_rec = float(recall_score(v_targets, v_preds, zero_division=0))
    v_f1 = float(f1_score(v_targets, v_preds, zero_division=0))
    v_auc = float(roc_auc_score(v_targets, v_probs)) if len(np.unique(v_targets)) > 1 else 0.0

    history["epoch"].append(epoch)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(v_loss)
    history["holdout_capture_5pct"].append(cap5)
    history["holdout_capture_10pct"].append(cap10)
    history["holdout_capture_20pct"].append(cap20)
    history["holdout_lift_5pct"].append(lift5)
    history["holdout_lift_10pct"].append(lift10)
    history["holdout_lift_20pct"].append(lift20)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(v_acc)
    history["train_precision"].append(tr_prec)
    history["val_precision"].append(v_prec)
    history["train_recall"].append(tr_rec)
    history["val_recall"].append(v_rec)
    history["train_f1"].append(tr_f1)
    history["val_f1"].append(v_f1)
    history["val_auc"].append(v_auc)

    # Chọn Best Epoch dựa trên Capture@10% & Capture@20% của tập Inner-Validation (thuộc Reference) + kiểm soát Val Loss
    composite_score = 0.6 * val_cap10 + 0.4 * val_cap20 - 0.15 * max(0.0, v_loss - tr_loss)
    if composite_score >= best_composite_score:
        best_composite_score = composite_score
        best_val_f1 = v_f1
        best_epoch = epoch
        best_holdout_probs = holdout_probs_ep.copy()
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "feature_columns": FEATURE_COLS,
                "scaler_mean": scaler.mean_.tolist(),
                "scaler_scale": scaler.scale_.tolist(),
                "val_metrics": {
                    "val_loss": v_loss,
                    "holdout_capture_5pct": cap5,
                    "holdout_capture_10pct": cap10,
                    "holdout_capture_20pct": cap20,
                    "holdout_lift_5pct": lift5,
                    "holdout_lift_10pct": lift10,
                    "holdout_lift_20pct": lift20,
                    "val_acc": v_acc,
                    "val_f1": v_f1,
                    "val_auc": v_auc,
                },
            },
            BEST_PT_PATH,
        )

print(
    f"\nHoàn tất huấn luyện! Best Model tại Epoch {best_epoch} (chọn theo PU Length Capture@K%):\n"
    f" - Holdout Capture@5%  = {history['holdout_capture_5pct'][best_epoch-1]*100:.2f}% (Lift@5%  = {history['holdout_lift_5pct'][best_epoch-1]:.2f}x)\n"
    f" - Holdout Capture@10% = {history['holdout_capture_10pct'][best_epoch-1]*100:.2f}% (Lift@10% = {history['holdout_lift_10pct'][best_epoch-1]:.2f}x)\n"
    f" - Holdout Capture@20% = {history['holdout_capture_20pct'][best_epoch-1]*100:.2f}% (Lift@20% = {history['holdout_lift_20pct'][best_epoch-1]:.2f}x)"
)

# Dự đoán xác suất ngập cho toàn bộ các đoạn đường TP.HCM bằng Best Model
checkpoint = torch.load(BEST_PT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
X_all = scaler.transform(clean_data[FEATURE_COLS].to_numpy(dtype=np.float32))
with torch.no_grad():
    all_probs = torch.sigmoid(model(torch.from_numpy(X_all).to(device))).cpu().numpy()
pred_df = pd.DataFrame(
    {
        "road_segment_id": clean_data["road_segment_id"],
        "partition": clean_data["partition"],
        "flood_susceptibility_prob": all_probs,
    }
)
PRED_PARQUET_PATH = RUN_DRIVE_DIR / "a100_road_flood_predictions.parquet"
pred_df.to_parquet(PRED_PARQUET_PATH, index=False)
(RUN_DRIVE_DIR / "training_history_metrics.json").write_text(
    json.dumps(history, indent=2), encoding="utf-8"
)

# Bổ sung đặc trưng địa hình cục bộ theo ô lưới 5km×5km (block_rel_elev_z, local_pocket_index) & Huấn luyện Logistic Regression L2 + Tri-Model Ensemble
from flood_ml.inference import compute_local_basin_features, fit_local_basin_stats
basin_stats = fit_local_basin_stats(train_raw)
clean_data = compute_local_basin_features(clean_data, basin_stats=basin_stats)
LOCAL_FEATURE_COLS = [*FEATURE_COLS, "block_rel_elev_z", "local_pocket_index"]
train_df_full_ref = build_balanced_split(train_raw, bg_ratio=3)
train_df_loc = train_df_full_ref[["road_segment_id"]].merge(clean_data[["road_segment_id", "block_rel_elev_z", "local_pocket_index"]], on="road_segment_id", how="left")
val_raw_loc = val_raw[["road_segment_id"]].merge(clean_data[["road_segment_id", *LOCAL_FEATURE_COLS]], on="road_segment_id", how="left")
scaler_lr_local = StandardScaler()
X_train_lr_loc = scaler_lr_local.fit_transform(
    pd.concat([train_df_full_ref[FEATURE_COLS].reset_index(drop=True), train_df_loc[["block_rel_elev_z", "local_pocket_index"]].reset_index(drop=True)], axis=1).to_numpy(dtype=np.float32)
)
clf_lr_ref = LogisticRegression(C=0.25, class_weight="balanced", max_iter=500, random_state=42)
clf_lr_ref.fit(X_train_lr_loc, train_df_full_ref["label"].to_numpy(dtype=np.int32))
holdout_lr_probs = clf_lr_ref.predict_proba(
    scaler_lr_local.transform(val_raw_loc[LOCAL_FEATURE_COLS].to_numpy(dtype=np.float32))
)[:, 1]
holdout_mcda_df = val_raw[["road_segment_id"]].merge(df_mcda[["road_segment_id", "score"]], on="road_segment_id", how="left")
holdout_mcda_scores = holdout_mcda_df["score"].fillna(0.0).to_numpy(dtype=np.float64)
holdout_tri_ens_scores = (
    0.45 * best_holdout_probs + 0.45 * holdout_lr_probs + 0.10 * (holdout_mcda_scores / 100.0)
)

# 2. Vẽ Dashboard 4 biểu đồ chuẩn Positive-Unlabeled (Loss, Capture@K%, Lift@K%, Gain Curve)
HISTORY_PNG = RUN_DRIVE_DIR / "training_history.png"
epochs_x = history["epoch"]
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# (1) Train Loss vs Val Loss
axes[0, 0].plot(epochs_x, history["train_loss"], label="Train Loss", color="#2878a5", linewidth=2.2)
axes[0, 0].plot(epochs_x, history["val_loss"], label="Val Loss (Holdout)", color="#c83e4d", linewidth=2.2)
axes[0, 0].axvline(best_epoch, color="#2b9348", linestyle="--", label=f"Best Epoch ({best_epoch})")
axes[0, 0].set_title("1. Biểu đồ Hàm mất mát (Train Loss vs Val Loss)", fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Smooth Weighted BCE Loss")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# (2) Holdout Capture@5%, Capture@10%, Capture@20% theo chiều dài đường qua các Epoch
axes[0, 1].plot(epochs_x, [x * 100 for x in history["holdout_capture_20pct"]], label="Capture@20% chiều dài đường", color="#2b9348", linewidth=2.3)
axes[0, 1].plot(epochs_x, [x * 100 for x in history["holdout_capture_10pct"]], label="Capture@10% chiều dài đường (Chính)", color="#2878a5", linewidth=2.4)
axes[0, 1].plot(epochs_x, [x * 100 for x in history["holdout_capture_5pct"]], label="Capture@5% chiều dài đường", color="#ef7c45", linewidth=2.0)
axes[0, 1].axvline(best_epoch, color="#2b9348", linestyle="--", label=f"Best Epoch ({best_epoch})")
axes[0, 1].set_title("2. Tỷ lệ Chiều dài Đường ngập bắt được trên Holdout (Capture@K%)", fontweight="bold")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Capture Rate theo chiều dài đường (%)")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# (3) Holdout Lift@5%, Lift@10%, Lift@20% qua các Epoch
axes[1, 0].plot(epochs_x, history["holdout_lift_5pct"], label="Lift@5% (Ngân sách hẹp)", color="#ef7c45", linewidth=2.2)
axes[1, 0].plot(epochs_x, history["holdout_lift_10pct"], label="Lift@10% (Thước đo chính)", color="#2878a5", linewidth=2.4)
axes[1, 0].plot(epochs_x, history["holdout_lift_20pct"], label="Lift@20% (Ngân sách mở rộng)", color="#2b9348", linewidth=2.2)
axes[1, 0].axhline(1.0, color="gray", linestyle=":", label="Ngẫu nhiên (Lift = 1.0x)")
axes[1, 0].axvline(best_epoch, color="#2b9348", linestyle="--")
axes[1, 0].set_title("3. Hệ số Khuếch đại Hiệu quả Xếp hạng (Lift@K% qua các Epoch)", fontweight="bold")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Hệ số Lift (gấp N lần chọn ngẫu nhiên)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# (4) Đường cong Gain Curve (Cumulative Length Capture) tại Best Epoch: MLP vs Logistic Regression vs MCDA
for label_m, arr_s, c_hex in (
    ("Tri-Model Ensemble (MLP + LogReg Local + MCDA)", holdout_tri_ens_scores, "#d90429"),
    (f"Deep MLP (Ablation - Best Ep {best_epoch})", best_holdout_probs, "#2b9348"),
    ("Logistic Regression + Local Basin (L2)", holdout_lr_probs, "#7b2cbf"),
    ("Baseline MCDA (Địa vật lý)", holdout_mcda_scores, "#2878a5"),
):
    if arr_s is not None:
        ord_i = np.argsort(-arr_s, kind="stable")
        c_road = np.cumsum(holdout_lengths[ord_i]) / total_holdout_len * 100.0
        c_flood = np.cumsum(holdout_lengths[ord_i] * holdout_labels[ord_i]) / flooded_holdout_len * 100.0
        axes[1, 1].plot(c_road, c_flood, label=label_m, color=c_hex, linewidth=2.3)
axes[1, 1].plot([0, 100], [0, 100], color="gray", linestyle="--", alpha=0.6, label="Ngẫu nhiên (Lift = 1.0x)")
axes[1, 1].axvline(10.0, color="#c83e4d", linestyle=":", label="Ngân sách Top 10%")
axes[1, 1].set_title(f"4. Đường cong Gain Curve theo Chiều dài Đường (Best Epoch {best_epoch})", fontweight="bold")
axes[1, 1].set_xlabel("% Chiều dài mạng lưới đường được ưu tiên kiểm tra (Top-K%)")
axes[1, 1].set_ylabel("% Chiều dài đường ngập thực tế bắt được (Capture Rate)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(loc="lower right")

fig.suptitle(
    f"Lịch sử Huấn luyện & Đánh giá Xếp hạng PU (Capture@K% / Lift@K%) trên {GPU_INFO} — Run: {RUN_ID}",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])
fig.savefig(HISTORY_PNG, dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

display(Markdown("### Biểu đồ Lịch sử Huấn luyện theo Thước đo Chính (`Loss`, `Capture@K%`, `Lift@K%` & `Gain Curve`)"))
display(Image(filename=str(HISTORY_PNG)))
for title, fname in (
    ("Bản đồ Điểm nhạy cảm ngập lụt (Baseline MCDA Scores)", "baseline_scores.png"),
    ("Phân phối các đặc trưng địa hình & thủy văn", "feature_distributions.png"),
    ("Bản đồ Phân vùng Không gian (Reference / Holdout / Guard)", "spatial_partitions.png"),
):
    img_path = ARTIFACTS_DIR / fname
    if img_path.is_file():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(img_path)))

## 8. Kiểm định Độ chính xác Xếp hạng Nguy cơ (`Capture@K%` & `Lift@K%` trên dữ liệu PU), Spatial Validation & Phân tích `Bias`
Đánh giá toàn diện theo 3 trụ cột khoa học và 2 nguyên tắc phương pháp luận (**Positive-Unlabeled Learning** & **Nonlinear Ablation Study**):
1. **Độ chính xác xếp hạng nguy cơ (`Capture@K%` & `Lift@K%` theo chiều dài đường — Thước đo chính cho dữ liệu PU)**: Do dữ liệu ngập báo chí là **Positive-Unlabeled (`PU`)** (các đoạn chưa có báo cáo là `unknown`, không phải `verified negative`), `Capture@K%` và `Lift@K%` theo chiều dài đường tại `Top 1%`, `Top 5%`, `Top 10%`, `Top 20%` là thước đo chính xác nhất, trong khi `PU-AUC` chỉ mang tính tham khảo. Đồng thời đối chiếu **`Deep MLP (Nonlinear Ablation)`** với mô hình tuyến tính ít tham số **`Logistic Regression (L2 — 9 tham số)`**, **`Hybrid Ensemble`** và **`Baseline MCDA`**.
2. **Spatial Validation (Kiểm định chéo theo từng Ô lưới Không gian 5km × 5km)**: Đánh giá `Capture@10%`, `Lift@10%` và tương quan `Spearman` trên từng khối không gian (`block_id`) độc lập trong vùng `Holdout` (đã cách ly bởi vùng đệm `guard = 1000m`).
3. **Kiểm tra Bias (Thiên lệch Báo cáo & Cấp hạng đường OSM `highway`)**: Đo lường mức độ thiên lệch báo chí (`Reporting Bias`) giữa trục đường chính (`primary/secondary`) và đường dân sinh (`residential/service`).


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, Markdown, display
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Tự động khôi phục các biến từ Google Drive nếu Colab vừa khởi động lại phiên (Restart session)
if "DRIVE_ROOT" not in globals():
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    DRIVE_ROOT = Path("/content/drive/MyDrive/hcmc-road-flood-ml")

if "CHECKOUT" not in globals():
    CHECKOUT = Path("/content/hcmc-road-flood-ml")

if "RUN_ID" not in globals():
    run_dirs = sorted(
        [d for d in (DRIVE_ROOT / "runs").glob("*") if d.is_dir() and (d / "a100_road_flood_predictions.parquet").is_file()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not run_dirs:
        run_dirs = sorted([d for d in (DRIVE_ROOT / "runs").glob("*") if d.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    RUN_ID = run_dirs[0].name
    print(f"Đã tự động khôi phục phiên làm việc từ Google Drive: RUN_ID = {RUN_ID}")

RUN_DRIVE_DIR = DRIVE_ROOT / "runs" / RUN_ID
PROCESSED_DIR = CHECKOUT / "data" / "processed" / RUN_ID
if not PROCESSED_DIR.is_dir():
    PROCESSED_DIR = RUN_DRIVE_DIR / "data" / "processed" / RUN_ID

FEATURE_COLS = [
    "elevation_median_m",
    "elevation_p10_m",
    "relative_elevation_m",
    "slope_mean_deg",
    "builtup_ratio_250m",
    "vegetation_ratio_250m",
    "dist_water_m",
    "water_area_ratio_250m",
]

if "clean_data" not in globals() or "df_part" not in globals() or "pred_df" not in globals():
    df_feat = pd.read_parquet(PROCESSED_DIR / "road_features.parquet")
    df_part = pd.read_parquet(PROCESSED_DIR / "road_partitions.parquet")
    df_confirmed = pd.read_parquet(PROCESSED_DIR / "confirmed_event_segments.parquet")
    matches_path = PROCESSED_DIR / "road_flood_matches.parquet"
    df_matches = pd.read_parquet(matches_path) if matches_path.is_file() else pd.DataFrame()
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df_feat.columns]
    if len(df_confirmed) > 0 and "road_segment_id" in df_confirmed.columns:
        pos_ids = set(df_confirmed["road_segment_id"].dropna().unique())
    elif len(df_matches) > 0 and "road_segment_id" in df_matches.columns:
        valid_matches = df_matches.dropna(subset=["road_segment_id"])
        if "match_distance_m" in valid_matches.columns:
            close_matches = valid_matches[valid_matches["match_distance_m"] <= 50.0]
            if len(close_matches) >= 50:
                valid_matches = close_matches
        pos_ids = set(valid_matches["road_segment_id"].unique())
    else:
        pos_ids = set()
    data = df_feat.merge(df_part[["road_segment_id", "partition"]], on="road_segment_id", how="inner")
    data["label"] = data["road_segment_id"].isin(pos_ids).astype(np.float32)
    valid_mask = data[FEATURE_COLS].notna().all(axis=1)
    if "score_eligible" in data.columns:
        valid_mask = valid_mask & data["score_eligible"].fillna(True)
    clean_data = data.loc[valid_mask].reset_index(drop=True)
    pred_df = pd.read_parquet(RUN_DRIVE_DIR / "a100_road_flood_predictions.parquet")

if "X_train" not in globals() or "y_train" not in globals() or "scaler" not in globals():
    rng_rec = np.random.default_rng(42)
    train_raw = clean_data[clean_data["partition"] == "reference"]
    pos_sub = train_raw[train_raw["label"] == 1.0]
    bg_sub = train_raw[train_raw["label"] == 0.0]
    n_bg_sub = min(len(bg_sub), len(pos_sub) * 3)
    train_df = pd.concat(
        [pos_sub, bg_sub.iloc[rng_rec.choice(len(bg_sub), size=n_bg_sub, replace=False)]],
        ignore_index=True,
    ).sample(frac=1.0, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURE_COLS].to_numpy(dtype=np.float32))
    y_train = train_df["label"].to_numpy(dtype=np.float32)

if "history" not in globals() or "best_epoch" not in globals():
    hist_file = RUN_DRIVE_DIR / "training_history_metrics.json"
    if hist_file.is_file():
        history = json.loads(hist_file.read_text(encoding="utf-8"))
        best_epoch = int(np.argmax(history.get("val_f1", [0.0])) + 1)
    else:
        history = {"train_loss": [0.0], "val_loss": [0.0]}
        best_epoch = 1

# Tổng hợp bảng đánh giá trên toàn bộ mạng lưới đường hợp lệ
df_mcda = pd.read_parquet(PROCESSED_DIR / "road_susceptibility.parquet")
base_eval = clean_data.copy()
if "highway" not in base_eval.columns:
    if "highway_type" in base_eval.columns:
        base_eval["highway"] = base_eval["highway_type"]
    else:
        roads_gpkg = CHECKOUT / "data" / "interim" / RUN_ID / "road_segments.gpkg"
        if not roads_gpkg.is_file():
            roads_gpkg = RUN_DRIVE_DIR / "data" / "interim" / RUN_ID / "road_segments.gpkg"
        if roads_gpkg.is_file():
            import geopandas as gpd
            r_meta = gpd.read_file(roads_gpkg, ignore_geometry=True)
            h_col = "highway_type" if "highway_type" in r_meta.columns else "highway"
            if h_col in r_meta.columns:
                base_eval = base_eval.merge(
                    r_meta[["road_segment_id", h_col]].rename(columns={h_col: "highway"}),
                    on="road_segment_id",
                    how="left",
                )
        if "highway" not in base_eval.columns:
            base_eval["highway"] = "residential"

if "length_m" not in base_eval.columns:
    if "road_length_m" in base_eval.columns:
        base_eval["length_m"] = base_eval["road_length_m"]
    elif "road_length_m" in df_mcda.columns:
        base_eval = base_eval.merge(
            df_mcda[["road_segment_id", "road_length_m"]].rename(columns={"road_length_m": "length_m"}),
            on="road_segment_id",
            how="left",
        )
    else:
        base_eval["length_m"] = 150.0

eval_df = (
    base_eval[
        [
            "road_segment_id",
            "partition",
            "label",
            "highway",
            "length_m",
            *FEATURE_COLS,
        ]
    ]
    .merge(df_part[["road_segment_id", "block_id"]], on="road_segment_id", how="left")
    .merge(df_mcda[["road_segment_id", "score"]], on="road_segment_id", how="left")
    .merge(pred_df[["road_segment_id", "flood_susceptibility_prob"]], on="road_segment_id", how="left")
)
eval_df = eval_df.dropna(subset=["score", "flood_susceptibility_prob", "length_m", *FEATURE_COLS]).reset_index(drop=True)

# Bổ sung đặc trưng địa hình cục bộ theo ô lưới 5km×5km (block_rel_elev_z, local_pocket_index) để nhận diện túi ngập ở vùng địa hình cao (như 121:240)
from flood_ml.inference import compute_local_basin_features, fit_local_basin_stats
ref_seed = eval_df[eval_df["partition"] == "reference"]
basin_stats = fit_local_basin_stats(ref_seed if len(ref_seed) > 0 else eval_df)
eval_df = compute_local_basin_features(eval_df, basin_stats=basin_stats)
LOCAL_FEATURE_COLS = [*FEATURE_COLS, "block_rel_elev_z", "local_pocket_index"]

rng_lr = np.random.default_rng(42)
ref_eval = eval_df[eval_df["partition"] == "reference"]
pos_eval = ref_eval[ref_eval["label"] == 1.0]
bg_eval = ref_eval[ref_eval["label"] == 0.0]
n_bg_eval = min(len(bg_eval), len(pos_eval) * 3)
tr_eval = pd.concat(
    [pos_eval, bg_eval.iloc[rng_lr.choice(len(bg_eval), size=n_bg_eval, replace=False)]],
    ignore_index=True,
).sample(frac=1.0, random_state=42)

# (a) Logistic Regression chuẩn hóa L2 (9 tham số cơ sở)
scaler_lr8 = StandardScaler()
X_tr8 = scaler_lr8.fit_transform(tr_eval[FEATURE_COLS].to_numpy(dtype=np.float32))
y_tr_eval = tr_eval["label"].to_numpy(dtype=np.int32)
clf_lr = LogisticRegression(C=0.25, class_weight="balanced", max_iter=500, random_state=42)
clf_lr.fit(X_tr8, y_tr_eval)
eval_df["logreg_prob"] = clf_lr.predict_proba(scaler_lr8.transform(eval_df[FEATURE_COLS].to_numpy(dtype=np.float32)))[:, 1]

# (b) Logistic Regression + Local Basin (L2 - 11 tham số có chuẩn hóa địa hình cục bộ theo ô lưới)
scaler_lr_local = StandardScaler()
X_tr10 = scaler_lr_local.fit_transform(tr_eval[LOCAL_FEATURE_COLS].to_numpy(dtype=np.float32))
clf_lr_local = LogisticRegression(C=0.25, class_weight="balanced", max_iter=500, random_state=42)
clf_lr_local.fit(X_tr10, y_tr_eval)
eval_df["logreg_local_prob"] = clf_lr_local.predict_proba(
    scaler_lr_local.transform(eval_df[LOCAL_FEATURE_COLS].to_numpy(dtype=np.float32))
)[:, 1]

eval_df["elev_risk_score"] = -eval_df["elevation_median_m"]
# (c) Tri-Model Ensemble (45% Deep MLP + 45% LogReg Local-Basin + 10% MCDA)
eval_df["hybrid_score"] = (
    0.45 * eval_df["flood_susceptibility_prob"]
    + 0.45 * eval_df["logreg_local_prob"]
    + 0.10 * (eval_df["score"] / 100.0)
)

holdout_df = eval_df[eval_df["partition"] == "holdout"].copy()
if (holdout_df["label"] == 1.0).sum() < 5:
    holdout_df = eval_df.copy()

total_holdout_len_m = float(holdout_df["length_m"].sum())
flooded_holdout_len_m = float(holdout_df.loc[holdout_df["label"] == 1.0, "length_m"].sum())

# ==============================================================================
# 1. ĐỘ CHÍNH XÁC XẾP HẠNG NGUY CƠ (TOP-K LENGTH CAPTURE & LIFT TRÊN DỮ LIỆU PU)
# ==============================================================================
TOP_K_BUDGETS = (0.01, 0.05, 0.10, 0.20)
METHODS = (
    ("Tri-Model Ensemble (MLP + LogReg Local + MCDA)", "hybrid_score"),
    ("Logistic Regression + Local Basin (L2 - 11 tham số)", "logreg_local_prob"),
    ("Deep MLP (Nonlinear Ablation)", "flood_susceptibility_prob"),
    ("Logistic Regression (L2 - 9 tham số)", "logreg_prob"),
    ("Baseline MCDA (Địa vật lý)", "score"),
    ("Elevation-Only (Độ cao thuần)", "elev_risk_score"),
)

ranking_rows = []
for method_label, col_name in METHODS:
    ordered = holdout_df.sort_values([col_name, "road_segment_id"], ascending=[False, True], kind="stable")
    cum_len = ordered["length_m"].cumsum().to_numpy()
    row_item = {"Phương pháp": method_label}
    for k_frac in TOP_K_BUDGETS:
        target_len = k_frac * total_holdout_len_m
        n_sel = min(int(np.searchsorted(cum_len, target_len, side="left")) + 1, len(ordered))
        sub = ordered.iloc[:n_sel]
        cap_len = float(sub.loc[sub["label"] == 1.0, "length_m"].sum())
        cap_rate = cap_len / max(flooded_holdout_len_m, 1e-6)
        actual_frac = float(sub["length_m"].sum()) / max(total_holdout_len_m, 1e-6)
        lift = cap_rate / max(actual_frac, 1e-6)
        pct = int(k_frac * 100)
        row_item[f"Capture@{pct}%"] = round(cap_rate, 4)
        row_item[f"Lift@{pct}%"] = round(lift, 2)
    ranking_rows.append(row_item)

from IPython.display import HTML
ranking_table = pd.DataFrame(ranking_rows)
display(Markdown("### 1. Bảng Độ chính xác Xếp hạng Nguy cơ (`Capture@K%` & `Lift@K%` theo chiều dài đường — Thước đo chính cho dữ liệu PU)"))
display(HTML(ranking_table.to_html(index=False)))

# ==============================================================================
# 2. SPATIAL VALIDATION (ĐÁNH GIÁ TRÊN TỪNG Ô LƯỚI KHÔNG GIAN 5KM × 5KM)
# ==============================================================================
block_rows = []
for block_id, grp in holdout_df.groupby("block_id", sort=True):
    n_roads = len(grp)
    n_pos = int((grp["label"] == 1.0).sum())
    if n_roads < 30:
        continue
    rho_mlp, _ = spearmanr(grp["flood_susceptibility_prob"], grp["score"])
    rho_ens, _ = spearmanr(grp["hybrid_score"], grp["score"])
    grp_total_len = max(float(grp["length_m"].sum()), 1e-6)
    grp_flood_len = max(float(grp.loc[grp["label"] == 1.0, "length_m"].sum()), 1e-6)

    def _calc_block_cap(df_b: pd.DataFrame, col_s: str, budget_f: float = 0.10):
        if n_pos == 0:
            return np.nan, np.nan
        ord_b = df_b.sort_values([col_s, "road_segment_id"], ascending=[False, True], kind="stable")
        c_len = ord_b["length_m"].cumsum().to_numpy()
        idx_c = min(int(np.searchsorted(c_len, budget_f * grp_total_len, side="left")) + 1, len(ord_b))
        sub_b = ord_b.iloc[:idx_c]
        cap_r = float(sub_b.loc[sub_b["label"] == 1.0, "length_m"].sum()) / grp_flood_len
        act_f = float(sub_b["length_m"].sum()) / grp_total_len
        return cap_r, cap_r / max(act_f, 1e-6)

    ens_cap10, ens_lift10 = _calc_block_cap(grp, "hybrid_score", 0.10)
    ens_cap20, _ = _calc_block_cap(grp, "hybrid_score", 0.20)
    mlp_cap10, mlp_lift10 = _calc_block_cap(grp, "flood_susceptibility_prob", 0.10)
    mlp_cap20, _ = _calc_block_cap(grp, "flood_susceptibility_prob", 0.20)
    lr_cap10, _ = _calc_block_cap(grp, "logreg_local_prob", 0.10)
    mcda_cap10, _ = _calc_block_cap(grp, "score", 0.10)

    block_rows.append(
        {
            "block_id": block_id,
            "Số đoạn đường": n_roads,
            "Số đoạn ngập": n_pos,
            "Ensemble Capture@10%": round(ens_cap10, 4) if np.isfinite(ens_cap10) else None,
            "Ensemble Lift@10%": round(ens_lift10, 2) if np.isfinite(ens_lift10) else None,
            "Ensemble Capture@20%": round(ens_cap20, 4) if np.isfinite(ens_cap20) else None,
            "MLP Capture@10%": round(mlp_cap10, 4) if np.isfinite(mlp_cap10) else None,
            "MLP Lift@10%": round(mlp_lift10, 2) if np.isfinite(mlp_lift10) else None,
            "MLP Capture@20%": round(mlp_cap20, 4) if np.isfinite(mlp_cap20) else None,
            "LogReg Capture@10%": round(lr_cap10, 4) if np.isfinite(lr_cap10) else None,
            "MCDA Capture@10%": round(mcda_cap10, 4) if np.isfinite(mcda_cap10) else None,
            "Tương quan Spearman (MLP vs MCDA)": round(float(rho_mlp), 4) if np.isfinite(rho_mlp) else None,
            "Tương quan Spearman (Ensemble vs MCDA)": round(float(rho_ens), 4) if np.isfinite(rho_ens) else None,
        }
    )

block_table = pd.DataFrame(block_rows).sort_values("Số đoạn ngập", ascending=False).reset_index(drop=True)
display(Markdown("### 2. Bảng Spatial Validation theo từng Khối Không gian 5km × 5km (`block_id` trong Holdout)"))
display(HTML(block_table.head(15).to_html(index=False)))

# ==============================================================================
# 3. KIỂM TRA BIAS (REPORTING BIAS & THIÊN LỆCH THEO CẤP HẠNG ĐƯỜNG HIGHWAY)
# ==============================================================================
eval_df["road_group"] = eval_df["highway"].map(
    lambda h: h if h in {"primary", "secondary", "tertiary", "residential", "service", "unclassified", "trunk"} else "other"
)

bias_rows = []
overall_pos_rate = float(eval_df["label"].mean())
for r_group, grp in eval_df.groupby("road_group"):
    if len(grp) < 50:
        continue
    obs_rate = float(grp["label"].mean())
    mlp_top10_rate = float((grp["flood_susceptibility_prob"] >= eval_df["flood_susceptibility_prob"].quantile(0.90)).mean())
    mcda_top10_rate = float((grp["score"] >= eval_df["score"].quantile(0.90)).mean())
    bias_rows.append(
        {
            "Cấp đường (highway)": r_group,
            "Số đoạn đường": len(grp),
            "Tỷ lệ báo cáo ngập (%)": round(obs_rate * 100, 2),
            "Hệ số Reporting Bias (so với TB)": round(obs_rate / max(overall_pos_rate, 1e-6), 2),
            "Điểm MLP TB": round(float(grp["flood_susceptibility_prob"].mean()), 4),
            "Điểm MCDA TB (0-100)": round(float(grp["score"].mean()), 2),
            "Tỷ lệ vào Top-10% Nguy cơ (MLP %)": round(mlp_top10_rate * 100, 2),
            "Tỷ lệ vào Top-10% Nguy cơ (MCDA %)": round(mcda_top10_rate * 100, 2),
        }
    )

bias_table = pd.DataFrame(bias_rows).sort_values("Hệ số Reporting Bias (so với TB)", ascending=False).reset_index(drop=True)
display(Markdown("### 3. Bảng Phân tích Thiên lệch Báo cáo (Reporting Bias) & Phân bổ Nguy cơ theo Cấp Đường"))
display(HTML(bias_table.to_html(index=False)))

# Vẽ Dashboard 3 biểu đồ: (1) Top-K Capture Curve, (2) Spatial Block Capture@10%, (3) Bias Analysis
AUDIT_PNG = RUN_DRIVE_DIR / "spatial_ranking_bias_audit.png"
fig, axes = plt.subplots(1, 3, figsize=(19, 5.8))

# Biểu đồ A: Đường cong Cumulative Length Capture (Gain Chart) trên Holdout
for method_label, col_name, color_hex in (
    ("Tri-Model Ensemble (MLP + LogReg Local + MCDA)", "hybrid_score", "#d90429"),
    ("Deep MLP (Nonlinear Ablation)", "flood_susceptibility_prob", "#2b9348"),
    ("Logistic Reg + Local Basin (L2)", "logreg_local_prob", "#7b2cbf"),
    ("Baseline MCDA", "score", "#2878a5"),
    ("Elevation-Only", "elev_risk_score", "#ef7c45"),
):
    ordered = holdout_df.sort_values([col_name, "road_segment_id"], ascending=[False, True], kind="stable")
    cum_road = ordered["length_m"].cumsum().to_numpy() / max(total_holdout_len_m, 1e-6)
    cum_flood = (ordered["length_m"] * ordered["label"]).cumsum().to_numpy() / max(flooded_holdout_len_m, 1e-6)
    axes[0].plot(cum_road * 100, cum_flood * 100, label=method_label, color=color_hex, linewidth=2.3)
axes[0].plot([0, 100], [0, 100], linestyle="--", color="gray", label="Ngẫu nhiên (Lift = 1.0x)")
axes[0].axvline(10, color="#c83e4d", linestyle=":", alpha=0.8, label="Ngân sách Top 10%")
axes[0].set_title("1. Độ chính xác Xếp hạng Nguy cơ (Gain Curve)", fontweight="bold")
axes[0].set_xlabel("% Chiều dài mạng lưới đường được ưu tiên kiểm tra (Top-K)")
axes[0].set_ylabel("% Chiều dài đường ngập thực tế bắt được (Capture Rate)")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="lower right")

# Biểu đồ B: Spatial Validation Capture@10% theo từng Khối Không gian 5x5km (chuẩn dữ liệu PU)
valid_blocks = block_table.dropna(subset=["MLP Capture@10%", "MCDA Capture@10%"]).head(10)
if len(valid_blocks) > 0:
    x_idx = np.arange(len(valid_blocks))
    width = 0.21
    axes[1].bar(x_idx - 1.5 * width, valid_blocks["Ensemble Capture@10%"] * 100, width, label="Tri-Model Ensemble", color="#d90429")
    axes[1].bar(x_idx - 0.5 * width, valid_blocks["MLP Capture@10%"] * 100, width, label="Deep MLP (Ablation)", color="#2b9348")
    axes[1].bar(x_idx + 0.5 * width, valid_blocks["LogReg Capture@10%"] * 100, width, label="LogReg + Local Basin", color="#7b2cbf")
    axes[1].bar(x_idx + 1.5 * width, valid_blocks["MCDA Capture@10%"] * 100, width, label="Baseline MCDA", color="#2878a5")
    axes[1].axhline(10.0, color="gray", linestyle="--", linewidth=1, label="Ngẫu nhiên (10%)")
    axes[1].set_xticks(x_idx)
    axes[1].set_xticklabels(valid_blocks["block_id"], rotation=30)
    axes[1].set_ylim(0.0, 105.0)
axes[1].set_title("2. Spatial Validation (Capture@10% qua các Ô lưới 5km × 5km)", fontweight="bold")
axes[1].set_xlabel("Mã Ô lưới Không gian độc lập (block_id)")
axes[1].set_ylabel("Capture@10% theo chiều dài đường ngập (%)")
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].legend(loc="upper right")

# Biểu đồ C: Kiểm tra Bias theo Cấp hạng đường (Reporting Bias vs Model Top-10% Selection)
x_b = np.arange(len(bias_table))
w_b = 0.26
axes[2].bar(x_b - w_b, bias_table["Tỷ lệ báo cáo ngập (%)"], w_b, label="Tỷ lệ báo cáo ngập thực tế (%)", color="#c83e4d")
axes[2].bar(x_b, bias_table["Tỷ lệ vào Top-10% Nguy cơ (MLP %)"], w_b, label="Top-10% Nguy cơ (Deep MLP %)", color="#2b9348")
axes[2].bar(x_b + w_b, bias_table["Tỷ lệ vào Top-10% Nguy cơ (MCDA %)"], w_b, label="Top-10% Nguy cơ (MCDA vật lý %)", color="#2878a5")
axes[2].set_xticks(x_b)
axes[2].set_xticklabels(bias_table["Cấp đường (highway)"], rotation=25)
axes[2].set_title("3. Phân tích Bias theo Cấp Đường (Reporting vs Model)", fontweight="bold")
axes[2].set_xlabel("Cấp hạng đường OSM (highway)")
axes[2].set_ylabel("Tỷ lệ (%)")
axes[2].grid(True, axis="y", alpha=0.3)
axes[2].legend()

fig.suptitle(
    f"Kiểm định Xếp hạng Nguy cơ (PU Capture@K), Spatial Validation & Phân tích Bias — Run: {RUN_ID}",
    fontsize=14,
    fontweight="bold",
    y=0.99,
)
fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.93])
fig.savefig(AUDIT_PNG, dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Lưu báo cáo JSON kiểm định chi tiết
audit_report = {
    "run_id": RUN_ID,
    "best_epoch": int(best_epoch),
    "overfit_gap_at_best_epoch": float(
        history["val_loss"][best_epoch - 1] - history["train_loss"][best_epoch - 1]
    ),
    "methodological_notes": {
        "pu_learning_evaluation": "Dữ liệu điểm ngập là Positive-Unlabeled (PU): nhãn 0 là unknown (chưa có báo cáo), không phải verified negative. Ưu tiên sử dụng Capture@K% và Lift@K% theo chiều dài đường làm thước đo chính; PU-AUC chỉ dùng để tham khảo.",
        "model_capacity_ablation": "Deep MLP được trình bày như thí nghiệm thăm dò tính phi tuyến (Nonlinear Ablation Study) để đối chiếu với Logistic Regression chuẩn hóa L2 (9 tham số) và Baseline MCDA.",
    },
    "risk_ranking_accuracy_holdout": ranking_rows,
    "spatial_validation_by_block": block_rows,
    "highway_bias_analysis": bias_rows,
}
(RUN_DRIVE_DIR / "spatial_ranking_bias_report.json").write_text(
    json.dumps(audit_report, indent=2, ensure_ascii=False), encoding="utf-8"
)
mean_block_cap10 = float(block_table["MLP Capture@10%"].dropna().mean()) if len(block_table) else 0.0
mean_ens_block_cap10 = float(block_table["Ensemble Capture@10%"].dropna().mean()) if len(block_table) else 0.0
mean_spearman = float(block_table["Tương quan Spearman (MLP vs MCDA)"].dropna().mean()) if len(block_table) else 0.0
rows_by_method = {r["Phương pháp"]: r for r in ranking_rows}
ens_row = rows_by_method.get("Tri-Model Ensemble (MLP + LogReg Local + MCDA)", ranking_rows[0])
lr_local_row = rows_by_method.get("Logistic Regression + Local Basin (L2 - 11 tham số)", ranking_rows[1])
mlp_row = rows_by_method.get("Deep MLP (Nonlinear Ablation)", ranking_rows[2])
lr_row = rows_by_method.get("Logistic Regression (L2 - 9 tham số)", ranking_rows[3])
display(
    Markdown(
        f"### Kết luận Kiểm định Mô hình (`{RUN_ID}`)\n"
        f"- **1. Độ chính xác xếp hạng nguy cơ trên dữ liệu Positive-Unlabeled (`Capture@K%` & `Lift@K%` theo chiều dài đường trên `Holdout`)**:\n"
        f"  - **Ngân sách Top 10% chiều dài đường**: `Tri-Model Ensemble (MLP + LogReg Local + MCDA)` bắt được **`{ens_row['Capture@10%']*100:.1f}%`** chiều dài đường ngập thực tế (**`Lift@10% = {ens_row['Lift@10%']:.2f}x`**); `Deep MLP (Nonlinear Ablation)` đạt **`{mlp_row['Capture@10%']*100:.1f}%`** (**`Lift@10% = {mlp_row['Lift@10%']:.2f}x`**); `Logistic Regression + Local Basin (L2 - 11 tham số)` đạt **`{lr_local_row['Capture@10%']*100:.1f}%`** (`Lift@10% = {lr_local_row['Lift@10%']:.2f}x`); `Logistic Regression (L2 - 9 tham số)` đạt **`{lr_row['Capture@10%']*100:.1f}%`** (`Lift@10% = {lr_row['Lift@10%']:.2f}x`).\n"
        f"  - **Ngân sách Top 20% chiều dài đường**: `Tri-Model Ensemble` bắt được **`{ens_row['Capture@20%']*100:.1f}%`** (**`Lift@20% = {ens_row['Lift@20%']:.2f}x`**), `Deep MLP` bắt được **`{mlp_row['Capture@20%']*100:.1f}%`** (**`Lift@20% = {mlp_row['Lift@20%']:.2f}x`**). *(Lưu ý phương pháp luận: Do nhãn `0` là `unknown / unlabeled background`, chỉ số `Capture@K%` và `Lift@K%` được dùng làm thước đo chính thay vì `ROC-AUC` truyền thống).*\n"
        f"- **2. Spatial Validation (Ô lưới 5km × 5km độc lập, cách ly `guard = 1000m`)**: Trung bình `Capture@10%` nội bộ từng ô lưới đạt **`{mean_ens_block_cap10*100:.1f}%`** (Ensemble) / **`{mean_block_cap10*100:.1f}%`** (Deep MLP), tương quan thứ hạng `Spearman` với chỉ số địa vật lý `MCDA` đạt **`{mean_spearman:.4f}`**.\n"
        f"- **3. Kiểm tra Bias (Reporting vs. Physical Susceptibility)**: Dữ liệu báo chí có thiên lệch báo cáo mạnh ở các trục đường chính (`primary`, `secondary`), trong khi mô hình giữ được sự cân bằng với quy luật địa hình - thủy văn của `MCDA`."
    )
)
display(Markdown("### Biểu đồ Kiểm định Xếp hạng Nguy cơ (PU Capture@K), Spatial Validation & Bias"))
display(Image(filename=str(AUDIT_PNG)))

## 9. Tính Điểm Nguy cơ Ngập cho Tất cả Đoạn đường trong Phạm vi Nghiên cứu TP.HCM (100% AOI)
Chấm điểm nguy cơ ngập (`mlp_risk_score_100`, `mcda_score_100`, `hybrid_risk_score_100`, `risk_percentile`, `risk_rank`, `risk_tier`) cho **toàn bộ 100% các đoạn đường trong phạm vi nghiên cứu TP.HCM (`road_features.parquet`)**, ghép tên đường thực tế từ OpenStreetMap (`road_name`, `highway_type`), xuất bảng tổng hợp theo cấp độ nguy cơ, Top 20 tuyến đường nguy cơ cao nhất TP.HCM và vẽ bản đồ nhiệt toàn thành phố.


In [ ]:
import geopandas as gpd
from IPython.display import HTML, Image, Markdown, display

# 1. Điền khuyết an toàn (nếu có) bằng median của tập Reference và dự đoán cho 100% đoạn đường trong AOI
all_features_df = df_feat.copy()
imputation_medians = {col: float(train_raw[col].median()) for col in FEATURE_COLS}
X_full_work = pd.DataFrame(index=all_features_df.index)
for col in FEATURE_COLS:
    X_full_work[col] = pd.to_numeric(all_features_df[col], errors="coerce").fillna(imputation_medians[col]).astype(np.float32)

X_full_scaled = scaler.transform(X_full_work[FEATURE_COLS].to_numpy(dtype=np.float32))
checkpoint_loaded = torch.load(BEST_PT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint_loaded["model_state_dict"])
model.eval()

full_probs_chunks = []
with torch.no_grad():
    for start_idx in range(0, len(X_full_scaled), 16384):
        batch_x = torch.from_numpy(X_full_scaled[start_idx : start_idx + 16384]).to(device)
        full_probs_chunks.append(torch.sigmoid(model(batch_x)).cpu().numpy())
full_probs = np.concatenate(full_probs_chunks)

# 2. Ghép thông tin phân vùng không gian, điểm MCDA và tên đường từ road_segments.gpkg
roads_gpkg = CHECKOUT / "data" / "interim" / RUN_ID / "road_segments.gpkg"
if not roads_gpkg.is_file():
    roads_gpkg = RUN_DRIVE_DIR / "data" / "interim" / RUN_ID / "road_segments.gpkg"
roads_gdf = gpd.read_file(roads_gpkg) if roads_gpkg.is_file() else None

full_scored = (
    all_features_df[["road_segment_id", *FEATURE_COLS, "score_eligible", "score_exclusion_reason"]]
    .merge(df_part[["road_segment_id", "partition", "block_id"]], on="road_segment_id", how="left")
    .merge(df_mcda[["road_segment_id", "score", "mcda_eligible", "score_reason"]], on="road_segment_id", how="left")
)
if roads_gdf is not None:
    meta_cols = [c for c in ("road_segment_id", "osm_way_id", "road_name", "highway_type", "length_m") if c in roads_gdf.columns]
    full_scored = full_scored.merge(roads_gdf[meta_cols], on="road_segment_id", how="left")

if "length_m" not in full_scored.columns:
    full_scored["length_m"] = all_features_df.get("road_length_m", pd.Series(150.0, index=all_features_df.index))
if "highway_type" not in full_scored.columns:
    full_scored["highway_type"] = all_features_df.get("highway_type", pd.Series("residential", index=all_features_df.index))
if "road_name" not in full_scored.columns:
    full_scored["road_name"] = None

# 3. Tính đặc trưng cục bộ theo ô lưới 5km×5km, xác suất LogReg Local-Basin & Bộ ba Điểm Nguy cơ Tri-Model Ensemble (0-100)
from flood_ml.inference import compute_local_basin_features, fit_local_basin_stats
full_basin_input = full_scored.copy()
for col in FEATURE_COLS:
    full_basin_input[col] = X_full_work[col]
ref_basin_seed = full_basin_input[full_basin_input["partition"] == "reference"]
basin_stats = fit_local_basin_stats(ref_basin_seed if len(ref_basin_seed) > 0 else full_basin_input)
full_basin_enriched = compute_local_basin_features(full_basin_input, basin_stats=basin_stats)
full_scored["block_rel_elev_z"] = full_basin_enriched["block_rel_elev_z"]
full_scored["local_pocket_index"] = full_basin_enriched["local_pocket_index"]

LOCAL_COLS_FULL = [*FEATURE_COLS, "block_rel_elev_z", "local_pocket_index"]
X_full_loc = pd.concat([X_full_work[FEATURE_COLS], full_scored[["block_rel_elev_z", "local_pocket_index"]]], axis=1)
if "clf_lr_local" in globals() and "scaler_lr_local" in globals():
    full_lr_probs = clf_lr_local.predict_proba(scaler_lr_local.transform(X_full_loc[LOCAL_COLS_FULL].to_numpy(dtype=np.float32)))[:, 1]
else:
    full_lr_probs = full_probs

full_scored["flood_susceptibility_prob"] = full_probs
full_scored["logreg_local_prob"] = full_lr_probs
full_scored["mlp_risk_score_100"] = (full_probs * 100.0).round(2)
full_scored["logreg_risk_score_100"] = (full_lr_probs * 100.0).round(2)
full_scored["mcda_score_100"] = pd.to_numeric(full_scored["score"], errors="coerce").round(2)
full_scored["hybrid_risk_score_100"] = np.where(
    full_scored["mcda_score_100"].notna(),
    0.45 * full_scored["mlp_risk_score_100"]
    + 0.45 * full_scored["logreg_risk_score_100"]
    + 0.10 * full_scored["mcda_score_100"],
    0.50 * full_scored["mlp_risk_score_100"] + 0.50 * full_scored["logreg_risk_score_100"],
).round(2)

eligible_mask = full_scored["score_eligible"].fillna(True)
full_scored["risk_percentile"] = np.nan
full_scored.loc[eligible_mask, "risk_percentile"] = (
    full_scored.loc[eligible_mask, "hybrid_risk_score_100"].rank(pct=True) * 100.0
).round(2)

pct = full_scored["risk_percentile"]
full_scored["risk_tier"] = np.where(
    ~eligible_mask,
    "7. Cầu/Hầm (Loại trừ cấu trúc)",
    np.where(
        pct >= 99.0,
        "1. Nguy kịch (Top 1%)",
        np.where(
            pct >= 95.0,
            "2. Rất cao (Top 5%)",
            np.where(
                pct >= 90.0,
                "3. Cao (Top 10%)",
                np.where(
                    pct >= 80.0,
                    "4. Trung bình - Cao (Top 20%)",
                    np.where(pct >= 50.0, "5. Trung bình", "6. Thấp"),
                ),
            ),
        ),
    ),
)

full_scored = full_scored.sort_values(
    ["score_eligible", "hybrid_risk_score_100", "mlp_risk_score_100"],
    ascending=[False, False, False],
    kind="stable",
).reset_index(drop=True)
full_scored["risk_rank"] = np.where(full_scored["score_eligible"].fillna(True), np.arange(1, len(full_scored) + 1), np.nan)

# Lưu toàn bộ bảng điểm nguy cơ ngập của 100% đoạn đường trong AOI
FULL_SCORES_PARQUET = RUN_DRIVE_DIR / "full_study_area_road_risk_scores.parquet"
TOP1000_CSV = RUN_DRIVE_DIR / "top_1000_highest_risk_roads.csv"
full_scored.to_parquet(FULL_SCORES_PARQUET, index=False)
full_scored[full_scored["score_eligible"].fillna(True)].head(1000).to_csv(TOP1000_CSV, index=False, encoding="utf-8-sig")

# 4. Bảng Thống kê Tổng hợp theo Phân tầng Nguy cơ trên toàn TP.HCM
tier_summary = (
    full_scored.groupby("risk_tier", as_index=False)
    .agg(
        so_doan_duong=("road_segment_id", "count"),
        tong_chieu_dai_km=("length_m", lambda s: round(float(s.sum()) / 1000.0, 2)),
        diem_mlp_tb_100=("mlp_risk_score_100", lambda s: round(float(s.mean()), 2)),
        diem_mcda_tb_100=("mcda_score_100", lambda s: round(float(s.dropna().mean()), 2) if len(s.dropna()) else None),
        diem_hybrid_tb_100=("hybrid_risk_score_100", lambda s: round(float(s.mean()), 2)),
    )
    .sort_values("risk_tier")
    .rename(
        columns={
            "risk_tier": "Phân tầng Nguy cơ Ngập",
            "so_doan_duong": "Số đoạn đường",
            "tong_chieu_dai_km": "Tổng chiều dài (km)",
            "diem_mlp_tb_100": "Điểm MLP TB (0-100)",
            "diem_mcda_tb_100": "Điểm MCDA TB (0-100)",
            "diem_hybrid_tb_100": "Điểm Hybrid TB (0-100)",
        }
    )
)
display(Markdown(f"### 1. Thống kê Điểm Nguy cơ Ngập cho Toàn bộ {len(full_scored):,} Đoạn đường trong Phạm vi Nghiên cứu TP.HCM"))
display(HTML(tier_summary.to_html(index=False)))

# 5. Bảng Top 20 Tuyến đường (có tên trên OSM) có Điểm Nguy cơ Ngập Cao nhất TP.HCM
named_roads = full_scored[full_scored["score_eligible"].fillna(True) & full_scored["road_name"].notna() & (full_scored["road_name"].astype(str).str.strip() != "")].copy()
if len(named_roads) > 0:
    top_streets = (
        named_roads.groupby(["road_name", "highway_type"], as_index=False)
        .agg(
            so_doan=("road_segment_id", "count"),
            chieu_dai_km=("length_m", lambda s: round(float(s.sum()) / 1000.0, 2)),
            diem_mlp_max=("mlp_risk_score_100", lambda s: round(float(s.max()), 2)),
            diem_mlp_tb=("mlp_risk_score_100", lambda s: round(float(s.mean()), 2)),
            diem_mcda_tb=("mcda_score_100", lambda s: round(float(s.dropna().mean()), 2) if len(s.dropna()) else None),
            diem_hybrid_max=("hybrid_risk_score_100", lambda s: round(float(s.max()), 2)),
            do_cao_tb_m=("elevation_median_m", lambda s: round(float(s.mean()), 2)),
            cach_kenh_rach_tb_m=("dist_water_m", lambda s: round(float(s.mean()), 1)),
        )
        .query("so_doan >= 3")
        .sort_values(["diem_hybrid_max", "diem_mlp_max", "diem_mlp_tb"], ascending=[False, False, False])
        .head(20)
        .rename(
            columns={
                "road_name": "Tên tuyến đường",
                "highway_type": "Cấp đường",
                "so_doan": "Số đoạn",
                "chieu_dai_km": "Chiều dài (km)",
                "diem_mlp_max": "Điểm MLP Max (0-100)",
                "diem_mlp_tb": "Điểm MLP TB (0-100)",
                "diem_mcda_tb": "Điểm MCDA TB (0-100)",
                "diem_hybrid_max": "Điểm Hybrid Max",
                "do_cao_tb_m": "Độ cao TB (m)",
                "cach_kenh_rach_tb_m": "Cách sông/kênh (m)",
            }
        )
    )
    display(Markdown("### 2. Top 20 Tuyến đường có Điểm Nguy cơ Ngập Cao nhất trong Phạm vi Nghiên cứu TP.HCM"))
    display(HTML(top_streets.to_html(index=False)))

# 6. Vẽ Bản đồ Nhiệt Điểm Nguy cơ Ngập toàn mạng lưới đường TP.HCM (MLP vs Hybrid)
FULL_MAP_PNG = RUN_DRIVE_DIR / "full_study_area_risk_map.png"
if roads_gdf is not None:
    aoi_gpkg = CHECKOUT / "data" / "interim" / RUN_ID / "study_area.gpkg"
    if not aoi_gpkg.is_file():
        aoi_gpkg = RUN_DRIVE_DIR / "data" / "interim" / RUN_ID / "study_area.gpkg"
    aoi_gdf = gpd.read_file(aoi_gpkg) if aoi_gpkg.is_file() else None
    map_gdf = roads_gdf[["road_segment_id", "geometry"]].merge(
        full_scored[["road_segment_id", "mlp_risk_score_100", "hybrid_risk_score_100", "risk_tier", "score_eligible"]],
        on="road_segment_id",
        how="inner",
    )
    map_gdf = map_gdf[map_gdf["score_eligible"].fillna(True)]
    fig, axes = plt.subplots(1, 2, figsize=(18, 9.5))
    for ax, col_plot, title_str in (
        (axes[0], "mlp_risk_score_100", "1. Bản đồ Điểm Nguy cơ Ngập AI — Deep MLP (0–100)"),
        (axes[1], "hybrid_risk_score_100", "2. Bản đồ Điểm Nguy cơ Ngập Tổng hợp — Tri-Model Ensemble (MLP + LogReg Local + MCDA)"),
    ):
        if aoi_gdf is not None:
            aoi_gdf.boundary.plot(ax=ax, color="#172b4d", linewidth=1.2)
        map_gdf.plot(
            ax=ax,
            column=col_plot,
            cmap="RdYlBu_r",
            linewidth=0.35,
            legend=True,
            legend_kwds={"label": "Chỉ số Nguy cơ Ngập (0 – 100)", "shrink": 0.72},
        )
        ax.set_title(title_str, fontweight="bold", fontsize=11.5, pad=12)
        ax.set_axis_off()
    fig.suptitle(
        f"Bản đồ Phân bố Nguy cơ Ngập lụt trên Toàn bộ {len(full_scored):,} Đoạn đường TP.HCM — Run: {RUN_ID}",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.94])
    fig.savefig(FULL_MAP_PNG, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Markdown("### 3. Bản đồ Nhiệt Điểm Nguy cơ Ngập Toàn Mạng lưới Đường TP.HCM"))
    display(Image(filename=str(FULL_MAP_PNG)))

    # 7. Xuất GeoJSON chuẩn Web GIS (WGS84 — EPSG:4326) ghép Điểm Nguy cơ với Tọa độ Đường + Bản đồ Tương tác HTML
    export_cols = [
        "road_segment_id",
        "road_name",
        "highway_type",
        "length_m",
        "mlp_risk_score_100",
        "logreg_risk_score_100",
        "mcda_score_100",
        "hybrid_risk_score_100",
        "risk_percentile",
        "risk_rank",
        "risk_tier",
        "elevation_median_m",
        "relative_elevation_m",
        "block_rel_elev_z",
        "local_pocket_index",
        "dist_water_m",
        "builtup_ratio_250m",
        "score_eligible",
    ]
    export_cols = [c for c in export_cols if c in full_scored.columns]
    geo_merged = roads_gdf[["road_segment_id", "geometry"]].merge(
        full_scored[export_cols], on="road_segment_id", how="inner"
    )
    # Chuyển hệ tọa độ từ VN-2000 (EPSG:9210) sang WGS84 (EPSG:4326: Kinh độ / Vĩ độ) chuẩn GeoJSON Web
    geo_wgs84 = geo_merged.to_crs("EPSG:4326")
    midpoints = geo_wgs84.geometry.interpolate(0.5, normalized=True)
    geo_wgs84["lon"] = midpoints.x.round(6)
    geo_wgs84["lat"] = midpoints.y.round(6)

    GEOJSON_TOP10_PATH = RUN_DRIVE_DIR / "hcmc_road_flood_risk_top10pct.geojson"
    GEOJSON_MAJOR_PATH = RUN_DRIVE_DIR / "hcmc_road_flood_risk_major_roads.geojson"
    GEOJSON_FULL_PATH = RUN_DRIVE_DIR / "hcmc_road_flood_risk_full.geojson"

    top10_gdf = geo_wgs84[geo_wgs84["score_eligible"].fillna(True) & (geo_wgs84["risk_percentile"] >= 90.0)]
    top10_gdf.to_file(GEOJSON_TOP10_PATH, driver="GeoJSON", coordinate_precision=6)

    major_gdf = geo_wgs84[
        geo_wgs84["highway_type"].isin(
            ["trunk", "trunk_link", "primary", "primary_link", "secondary", "secondary_link", "tertiary", "tertiary_link"]
        )
    ]
    major_gdf.to_file(GEOJSON_MAJOR_PATH, driver="GeoJSON", coordinate_precision=6)
    geo_wgs84.to_file(GEOJSON_FULL_PATH, driver="GeoJSON", coordinate_precision=6)

    # Tạo Bản đồ Tương tác HTML (Leaflet.js) hiển thị trực tiếp các tuyến đường nguy cơ cao trên nền bản đồ TP.HCM
    top_map_gdf = geo_wgs84[geo_wgs84["score_eligible"].fillna(True)].sort_values(["hybrid_risk_score_100", "mlp_risk_score_100"], ascending=[False, False]).head(5000)
    geojson_str = top_map_gdf.to_json()
    INTERACTIVE_HTML_PATH = RUN_DRIVE_DIR / "hcmc_interactive_flood_map.html"
    leaflet_html = f"""<!DOCTYPE html>
<html lang="vi">
<head>
  <meta charset="utf-8" />
  <title>Bản đồ Nguy cơ Ngập lụt Đường bộ TP.HCM — {RUN_ID}</title>
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    body {{ margin: 0; padding: 0; font-family: system-ui, -apple-system, sans-serif; }}
    #map {{ width: 100%; height: 650px; }}
    .legend {{ background: rgba(255,255,255,0.95); padding: 12px 14px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); line-height: 1.6; font-size: 13px; }}
    .legend i {{ width: 16px; height: 4px; display: inline-block; margin-right: 8px; vertical-align: middle; }}
  </style>
</head>
<body>
<div id="map"></div>
<script>
  const map = L.map('map', {{ preferCanvas: true }}).setView([10.7769, 106.7009], 12);
  L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
    attribution: '&copy; OpenStreetMap &copy; CARTO',
    maxZoom: 19
  }}).addTo(map);
  const roadData = {geojson_str};
  function getColor(score) {{
    return score >= 52 ? '#ff1744' :
           score >= 45 ? '#ff9100' :
           score >= 38 ? '#ffea00' : '#00e5ff';
  }}
  L.geoJSON(roadData, {{
    style: function(feature) {{
      const s = feature.properties.mlp_risk_score_100 || 0;
      return {{ color: getColor(s), weight: s >= 52 ? 4 : 2.5, opacity: 0.9 }};
    }},
    onEachFeature: function(feature, layer) {{
      const p = feature.properties;
      layer.bindPopup(
        `<b>Đường: ${{p.road_name || 'Đường chưa đặt tên'}}</b><br>` +
        `Cấp đường: <code>${{p.highway_type}}</code> | Dài: ${{Number(p.length_m || 0).toFixed(1)}} m<br>` +
        `<b>Điểm Nguy cơ AI (MLP): ${{Number(p.mlp_risk_score_100).toFixed(2)}} / 100</b><br>` +
        `Điểm MCDA Vật lý: ${{Number(p.mcda_score_100 || 0).toFixed(2)}} / 100<br>` +
        `Điểm Hybrid: ${{Number(p.hybrid_risk_score_100 || 0).toFixed(2)}} / 100<br>` +
        `Phân tầng: <b>${{p.risk_tier}}</b> (Hạng #${{p.risk_rank}})<br>` +
        `Độ cao TB: ${{Number(p.elevation_median_m || 0).toFixed(2)}} m | Cách sông/kênh: ${{Number(p.dist_water_m || 0).toFixed(1)}} m<br>` +
        `Tọa độ: ${{p.lat}}, ${{p.lon}}`
      );
    }}
  }}).addTo(map);
  const legend = L.control({{position: 'bottomright'}});
  legend.onAdd = function() {{
    const div = L.DomUtil.create('div', 'legend');
    div.innerHTML = '<b>Mức độ Nguy cơ Ngập (Top 5.000 đoạn)</b><br>' +
      '<i style="background:#ff1744"></i> Nguy kịch / Rất cao (≥ 52 điểm)<br>' +
      '<i style="background:#ff9100"></i> Cao (45 – 52 điểm)<br>' +
      '<i style="background:#ffea00"></i> Trung bình - Cao (38 – 45 điểm)<br>' +
      '<i style="background:#00e5ff"></i> Trung bình (< 38 điểm)';
    return div;
  }};
  legend.addTo(map);
</script>
</body>
</html>"""
    INTERACTIVE_HTML_PATH.write_text(leaflet_html, encoding="utf-8")
    print(f"Đã xuất 3 file GeoJSON (hệ tọa độ chuẩn WGS84 EPSG:4326) & Bản đồ HTML tương tác tại: {RUN_DRIVE_DIR}")
    print(f" - 1. Top 10% Nguy cơ cao nhất: {GEOJSON_TOP10_PATH.name} ({GEOJSON_TOP10_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
    print(f" - 2. Trục đường chính TP.HCM: {GEOJSON_MAJOR_PATH.name} ({GEOJSON_MAJOR_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
    print(f" - 3. Toàn bộ 100% đoạn đường: {GEOJSON_FULL_PATH.name} ({GEOJSON_FULL_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
    print(f" - 4. Bản đồ Web tương tác:    {INTERACTIVE_HTML_PATH.name}")

## 10. Lưu Model + Preprocessing + Config & Tải về Best Model (`best_model_<RUN_ID>.zip`)
Đóng gói mô hình tốt nhất (`best_model_a100.pt`, `flood_susceptibility_torchscript.pt`, `scaler.joblib`, `preprocessing_config.json`, `model_config.json`, `best_track_b_model.joblib`, bảng điểm nguy cơ & GeoJSON) thành 1 file `.zip` duy nhất và **tự động tải thẳng về máy tính**.


In [ ]:
import joblib

# 1. Tạo thư mục chuẩn lưu Model + Preprocessing + Config trên Google Drive
DEPLOY_DIR = RUN_DRIVE_DIR / "deployment_bundle"
(DEPLOY_DIR / "model").mkdir(parents=True, exist_ok=True)
(DEPLOY_DIR / "preprocessing").mkdir(parents=True, exist_ok=True)
(DEPLOY_DIR / "config").mkdir(parents=True, exist_ok=True)

# 2. Lưu bộ tiền xử lý (StandardScaler + Imputation Medians + Feature Schema)
SCALER_JOBLIB_PATH = DEPLOY_DIR / "preprocessing" / "scaler.joblib"
joblib.dump(scaler, SCALER_JOBLIB_PATH)

imputation_medians = {
    col: float(train_raw[col].median()) for col in FEATURE_COLS
}
feature_ranges = {
    col: {
        "min": float(clean_data[col].min()),
        "q05": float(clean_data[col].quantile(0.05)),
        "median": float(clean_data[col].median()),
        "q95": float(clean_data[col].quantile(0.95)),
        "max": float(clean_data[col].max()),
    }
    for col in FEATURE_COLS
}
if "basin_stats" not in globals():
    from flood_ml.inference import fit_local_basin_stats
    basin_stats = fit_local_basin_stats(train_raw)
prep_config = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "feature_columns": FEATURE_COLS,
    "scaler_type": "sklearn.preprocessing.StandardScaler",
    "scaler_mean": [float(x) for x in scaler.mean_],
    "scaler_scale": [float(x) for x in scaler.scale_],
    "imputation_medians": imputation_medians,
    "feature_distributions": feature_ranges,
    "basin_stats": basin_stats,
}
PREP_CONFIG_PATH = DEPLOY_DIR / "preprocessing" / "preprocessing_config.json"
PREP_CONFIG_PATH.write_text(json.dumps(prep_config, indent=2, ensure_ascii=False), encoding="utf-8")

# 3. Lưu Cấu hình Mô hình & Ngưỡng Phân tầng Nguy cơ (Thresholds)
probs_series = pred_df["flood_susceptibility_prob"]
thresholds_cfg = {
    "default": 0.5,
    "top_1_pct": float(probs_series.quantile(0.99)),
    "top_5_pct": float(probs_series.quantile(0.95)),
    "top_10_pct": float(probs_series.quantile(0.90)),
    "top_20_pct": float(probs_series.quantile(0.80)),
}
model_config = {
    "schema_version": 1,
    "model_name": "FloodSusceptibilityNet",
    "run_id": RUN_ID,
    "best_epoch": int(best_epoch),
    "hardware": GPU_INFO,
    "architecture": {
        "in_features": len(FEATURE_COLS),
        "hidden_dim": 64,
        "head_dim": 32,
        "dropout": 0.35,
        "residual_scale": 0.5,
        "activation": "SiLU",
        "normalization": "LayerNorm",
    },
    "training_hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": 4e-4,
        "weight_decay": 2e-2,
        "label_smoothing": 0.06,
        "input_noise_std": 0.05,
        "pos_weight": float(pos_weight_val),
    },
    "thresholds": thresholds_cfg,
    "validation_metrics_at_best_epoch": {
        "val_loss": float(history["val_loss"][best_epoch - 1]),
        "val_acc": float(history["val_acc"][best_epoch - 1]),
        "val_precision": float(history["val_precision"][best_epoch - 1]),
        "val_recall": float(history["val_recall"][best_epoch - 1]),
        "val_f1": float(history["val_f1"][best_epoch - 1]),
        "val_roc_auc": float(history["val_auc"][best_epoch - 1]),
    },
}
MODEL_CONFIG_PATH = DEPLOY_DIR / "config" / "model_config.json"
MODEL_CONFIG_PATH.write_text(json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8")

# 4. Lưu Checkpoint PyTorch đầy đủ + TorchScript (chạy độc lập không cần class)
full_checkpoint = {
    "epoch": int(best_epoch),
    "model_state_dict": model.state_dict(),
    "feature_columns": FEATURE_COLS,
    "scaler_mean": [float(x) for x in scaler.mean_],
    "scaler_scale": [float(x) for x in scaler.scale_],
    "imputation_medians": imputation_medians,
    "basin_stats": basin_stats,
    "thresholds": thresholds_cfg,
    "model_config": model_config,
    "val_metrics": model_config["validation_metrics_at_best_epoch"],
}
torch.save(full_checkpoint, BEST_PT_PATH)
torch.save(full_checkpoint, DEPLOY_DIR / "model" / "best_model_a100.pt")

import copy
model_cpu = copy.deepcopy(model).cpu().eval()
example_input = torch.zeros(1, len(FEATURE_COLS), dtype=torch.float32)
traced_model = torch.jit.trace(model_cpu, example_input)
TORCHSCRIPT_PATH = DEPLOY_DIR / "model" / "flood_susceptibility_torchscript.pt"
traced_model.save(str(TORCHSCRIPT_PATH))

# Copy thêm cấu hình MCDA chuẩn vào config/
for cfg_file in ("normalizer.json", "mcda_rules.json"):
    src_cfg = ARTIFACTS_DIR / cfg_file
    if src_cfg.is_file():
        shutil.copy2(src_cfg, DEPLOY_DIR / "config" / cfg_file)

BEST_BUNDLE_ZIP = DRIVE_ROOT / "runs" / RUN_ID / f"best_model_{RUN_ID}.zip"
MODELS_DIR = ARTIFACTS_DIR / "track_b_models"

selected_best_model = None
if MODELS_DIR.is_dir():
    preferred = MODELS_DIR / "track_b_r10_s42.joblib"
    all_models = sorted(MODELS_DIR.glob("*.joblib"))
    if preferred.is_file():
        selected_best_model = preferred
    elif all_models:
        selected_best_model = all_models[0]

with zipfile.ZipFile(BEST_BUNDLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if BEST_PT_PATH.is_file():
        zf.write(BEST_PT_PATH, arcname="model/best_model_a100.pt")
    if TORCHSCRIPT_PATH.is_file():
        zf.write(TORCHSCRIPT_PATH, arcname="model/flood_susceptibility_torchscript.pt")
    if SCALER_JOBLIB_PATH.is_file():
        zf.write(SCALER_JOBLIB_PATH, arcname="preprocessing/scaler.joblib")
    if PREP_CONFIG_PATH.is_file():
        zf.write(PREP_CONFIG_PATH, arcname="preprocessing/preprocessing_config.json")
    if MODEL_CONFIG_PATH.is_file():
        zf.write(MODEL_CONFIG_PATH, arcname="config/model_config.json")
    for cfg_file in ("normalizer.json", "mcda_rules.json"):
        cp = DEPLOY_DIR / "config" / cfg_file
        if cp.is_file():
            zf.write(cp, arcname=f"config/{cfg_file}")
    inf_py = CHECKOUT / "src" / "flood_ml" / "inference.py"
    if inf_py.is_file():
        zf.write(inf_py, arcname="inference.py")
    if PRED_PARQUET_PATH.is_file():
        zf.write(PRED_PARQUET_PATH, arcname="scores/a100_road_flood_predictions.parquet")
    for full_score_f in (
        "full_study_area_road_risk_scores.parquet",
        "top_1000_highest_risk_roads.csv",
        "hcmc_road_flood_risk_top10pct.geojson",
        "hcmc_road_flood_risk_major_roads.geojson",
        "hcmc_interactive_flood_map.html",
    ):
        fsp = RUN_DRIVE_DIR / full_score_f
        if fsp.is_file():
            zf.write(fsp, arcname=f"scores/{full_score_f}")
    if (RUN_DRIVE_DIR / "full_study_area_risk_map.png").is_file():
        zf.write(
            RUN_DRIVE_DIR / "full_study_area_risk_map.png",
            arcname="artifacts/full_study_area_risk_map.png",
        )
    if (RUN_DRIVE_DIR / "training_history_metrics.json").is_file():
        zf.write(
            RUN_DRIVE_DIR / "training_history_metrics.json",
            arcname="reports/training_history_metrics.json",
        )
    if selected_best_model is not None:
        zf.write(selected_best_model, arcname="best_track_b_model.joblib")
        for mf in sorted(MODELS_DIR.glob("*.joblib")):
            zf.write(mf, arcname=f"track_b_models/{mf.name}")
    for item in sorted(ARTIFACTS_DIR.glob("*")):
        if item.is_file():
            zf.write(item, arcname=f"artifacts/{item.name}")
    for score_file in ("road_susceptibility.parquet", "track_b_scores.parquet"):
        sp = PROCESSED_DIR / score_file
        if sp.is_file():
            zf.write(sp, arcname=f"scores/{score_file}")
    for rep_file in (
        "pipeline_report.html",
        "pipeline_report.json",
        "track_b.json",
        "evaluation.json",
        "colab_summary.md",
    ):
        rp = REPORTS / rep_file
        if rp.is_file():
            zf.write(rp, arcname=f"reports/{rep_file}")
    for png_path in (HISTORY_PNG, RUN_DRIVE_DIR / "spatial_ranking_bias_audit.png"):
        if png_path.is_file():
            zf.write(png_path, arcname=f"artifacts/{png_path.name}")
    if (RUN_DRIVE_DIR / "spatial_ranking_bias_report.json").is_file():
        zf.write(
            RUN_DRIVE_DIR / "spatial_ranking_bias_report.json",
            arcname="reports/spatial_ranking_bias_report.json",
        )

print(
    f"Đã đóng gói Best Model tại: {BEST_BUNDLE_ZIP} "
    f"({BEST_BUNDLE_ZIP.stat().st_size / 1024 / 1024:.2f} MB)"
)
print(f"- Best PyTorch A100 Model (Epoch {best_epoch}, F1={best_val_f1:.4f}): best_model_a100.pt")

try:
    from google.colab import files

    print("Đang tải file zip Best Model trực tiếp về trình duyệt của bạn...")
    files.download(str(BEST_BUNDLE_ZIP))
except Exception:
    print(f"Bạn có thể tải trực tiếp file từ Google Drive tại: {BEST_BUNDLE_ZIP}")

## 11. Đồng bộ Lịch sử Huấn luyện, Bảng Đánh giá, Bản đồ & Model Artifact lên Weights & Biases (`wandb`)
Đẩy toàn bộ kết quả thực nghiệm (`30 Epochs Loss/Acc/F1/AUC`, bảng `Capture@K / Lift@K`, `Spatial Validation 5km×5km`, `Reporting Bias`, các biểu đồ PNG, bản đồ tương tác Leaflet HTML và `Model Artifact`) lên dashboard **Weights & Biases (`hcmc-road-flood-ml`)**.
- Tự động lấy `WANDB_API_KEY` từ **Colab Secrets** (biểu tượng chìa khóa 🔑 bên trái) hoặc hiển thị ô nhập API Key trực tiếp nếu chưa cấu hình.


In [ ]:
import os
import subprocess
import sys

# 1. Cài đặt wandb (nếu chưa có trên runtime Colab)
try:
    import wandb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

# 2. Đăng nhập W&B: ưu tiên biến môi trường -> Colab Secrets (WANDB_API_KEY) -> nhập trực tiếp
wandb_key = os.environ.get("WANDB_API_KEY", "").strip()
if not wandb_key:
    try:
        from google.colab import userdata
        wandb_key = (userdata.get("WANDB_API_KEY") or "").strip()
    except Exception:
        wandb_key = ""

if not wandb_key:
    try:
        from getpass import getpass
        wandb_key = getpass("Dán trực tiếp WANDB_API_KEY vào ô này rồi bấm Enter (lấy tại https://wandb.ai/authorize): ").strip()
    except Exception:
        wandb_key = ""

wandb_mode = "online"
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    print("Không có WANDB_API_KEY; chuyển sang chế độ lưu W&B cục bộ (offline mode).")
    wandb_mode = "offline"

# 3. Khởi tạo W&B Run cho dự án hcmc-road-flood-ml
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "hcmc-road-flood-ml").strip() or "hcmc-road-flood-ml"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

run_config = {
    "run_id": RUN_ID,
    "hardware": GPU_INFO,
    "model_name": "FloodSusceptibilityNet",
    "feature_columns": FEATURE_COLS,
    "num_features": len(FEATURE_COLS),
    "hidden_dim": 64,
    "head_dim": 32,
    "dropout": 0.35,
    "residual_scale": 0.5,
    "batch_size": int(BATCH_SIZE),
    "epochs": int(EPOCHS),
    "learning_rate": 4e-4,
    "weight_decay": 2e-2,
    "label_smoothing": float(LABEL_SMOOTHING),
    "input_noise_std": float(INPUT_NOISE_STD),
    "pos_weight": float(pos_weight_val),
    "best_epoch": int(best_epoch),
}

wb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY if wandb_mode == "online" else None,
    name=f"{RUN_ID}-a100",
    id=f"{RUN_ID}-a100".replace("_", "-"),
    resume="allow",
    mode=wandb_mode,
    config=run_config,
    tags=["hcmc-flood", "a100", "deep-mlp", "spatial-validation", "anti-overfit"],
)

# 4. Log lịch sử huấn luyện từng Epoch (1 -> 30) để vẽ biểu đồ tương tác trên W&B
for i in range(len(history["epoch"])):
    ep = int(history["epoch"][i])
    tr_l = float(history["train_loss"][i])
    va_l = float(history["val_loss"][i])
    wandb.log(
        {
            "epoch": ep,
            "train/loss": tr_l,
            "val/loss": va_l,
            "train/accuracy": float(history["train_acc"][i]),
            "val/accuracy": float(history["val_acc"][i]),
            "train/precision": float(history["train_precision"][i]),
            "val/precision": float(history["val_precision"][i]),
            "train/recall": float(history["train_recall"][i]),
            "val/recall": float(history["val_recall"][i]),
            "train/f1": float(history["train_f1"][i]),
            "val/f1": float(history["val_f1"][i]),
            "val/roc_auc": float(history["val_auc"][i]),
            "diagnostics/overfit_loss_gap": va_l - tr_l,
        },
        step=ep,
    )

# 5. Log các Bảng Đánh giá (W&B Interactive Tables)
wb_tables = {}
if "ranking_table" in globals() and isinstance(ranking_table, pd.DataFrame):
    wb_tables["evaluation/risk_ranking_table"] = wandb.Table(dataframe=ranking_table)
if "block_table" in globals() and isinstance(block_table, pd.DataFrame):
    wb_tables["evaluation/spatial_validation_blocks"] = wandb.Table(dataframe=block_table)
if "bias_table" in globals() and isinstance(bias_table, pd.DataFrame):
    wb_tables["evaluation/highway_bias_analysis"] = wandb.Table(dataframe=bias_table)
if "tier_summary" in globals() and isinstance(tier_summary, pd.DataFrame):
    wb_tables["study_area/risk_tier_summary"] = wandb.Table(dataframe=tier_summary)
if "top_streets" in globals() and isinstance(top_streets, pd.DataFrame):
    wb_tables["study_area/top_20_highest_risk_streets"] = wandb.Table(dataframe=top_streets)

# 6. Log các Biểu đồ & Bản đồ (W&B Images + Interactive HTML Map)
wb_media = {}
for key_name, img_p in (
    ("charts/training_history", RUN_DRIVE_DIR / "training_history.png"),
    ("charts/spatial_ranking_bias_audit", RUN_DRIVE_DIR / "spatial_ranking_bias_audit.png"),
    ("maps/full_study_area_risk_map", RUN_DRIVE_DIR / "full_study_area_risk_map.png"),
    ("maps/baseline_mcda_scores", ARTIFACTS_DIR / "baseline_scores.png"),
    ("maps/feature_distributions", ARTIFACTS_DIR / "feature_distributions.png"),
    ("maps/spatial_partitions", ARTIFACTS_DIR / "spatial_partitions.png"),
):
    if img_p.is_file():
        wb_media[key_name] = wandb.Image(str(img_p), caption=f"{key_name} ({RUN_ID})")

html_map_file = RUN_DRIVE_DIR / "hcmc_interactive_flood_map.html"
if html_map_file.is_file():
    wb_media["maps/interactive_leaflet_map"] = wandb.Html(html_map_file.read_text(encoding="utf-8"))

if wb_tables or wb_media:
    wandb.log({**wb_tables, **wb_media})

# 7. Ghi Summary Metrics tốt nhất lên đầu trang Overview của W&B
wb_run.summary["best_epoch"] = int(best_epoch)
wb_run.summary["best_val_loss"] = float(history["val_loss"][best_epoch - 1])
wb_run.summary["best_val_accuracy"] = float(history["val_acc"][best_epoch - 1])
wb_run.summary["best_val_f1"] = float(history["val_f1"][best_epoch - 1])
wb_run.summary["best_val_roc_auc"] = float(history["val_auc"][best_epoch - 1])

if "ranking_rows" in globals() and len(ranking_rows) > 0:
    rows_by_m = {r["Phương pháp"]: r for r in ranking_rows}
    ens_r = rows_by_m.get("Tri-Model Ensemble (MLP + LogReg Local + MCDA)", ranking_rows[0])
    mlp_r = rows_by_m.get("Deep MLP (Nonlinear Ablation)", ranking_rows[min(2, len(ranking_rows) - 1)])
    wb_run.summary["capture_at_5pct"] = float(ens_r.get("Capture@5%", 0.0))
    wb_run.summary["lift_at_5pct"] = float(ens_r.get("Lift@5%", 0.0))
    wb_run.summary["capture_at_10pct"] = float(ens_r.get("Capture@10%", 0.0))
    wb_run.summary["lift_at_10pct"] = float(ens_r.get("Lift@10%", 0.0))
    wb_run.summary["capture_at_20pct"] = float(ens_r.get("Capture@20%", 0.0))
    wb_run.summary["lift_at_20pct"] = float(ens_r.get("Lift@20%", 0.0))
    wb_run.summary["mlp_capture_at_10pct"] = float(mlp_r.get("Capture@10%", 0.0))
    wb_run.summary["mlp_lift_at_10pct"] = float(mlp_r.get("Lift@10%", 0.0))
    wb_run.summary["holdout_pu_auc_ref"] = float(mlp_r.get("PU-AUC (Tham khảo)", mlp_r.get("Holdout ROC-AUC", 0.0)))

if "block_table" in globals() and isinstance(block_table, pd.DataFrame) and len(block_table) > 0:
    if "MLP Capture@10%" in block_table.columns:
        wb_run.summary["spatial_mean_block_capture10"] = float(block_table["MLP Capture@10%"].dropna().mean())
    if "MLP PU-AUC (Tham khảo)" in block_table.columns:
        wb_run.summary["spatial_mean_block_pu_auc_ref"] = float(block_table["MLP PU-AUC (Tham khảo)"].dropna().mean())
    elif "MLP ROC-AUC" in block_table.columns:
        wb_run.summary["spatial_mean_block_pu_auc_ref"] = float(block_table["MLP ROC-AUC"].dropna().mean())

# 8. Upload Model + Preprocessing + Config & GeoJSON lên W&B Artifacts (Quản lý phiên bản Model)
model_artifact = wandb.Artifact(
    name="hcmc-flood-susceptibility-model",
    type="model",
    description=f"FloodSusceptibilityNet trained on {GPU_INFO} (Run {RUN_ID}, Best Epoch {best_epoch})",
    metadata=run_config,
)
for candidate in (
    BEST_PT_PATH,
    RUN_DRIVE_DIR / "deployment_bundle" / "model" / "flood_susceptibility_torchscript.pt",
    RUN_DRIVE_DIR / "deployment_bundle" / "preprocessing" / "scaler.joblib",
    RUN_DRIVE_DIR / "deployment_bundle" / "preprocessing" / "preprocessing_config.json",
    RUN_DRIVE_DIR / "deployment_bundle" / "config" / "model_config.json",
    RUN_DRIVE_DIR / "spatial_ranking_bias_report.json",
    RUN_DRIVE_DIR / "top_1000_highest_risk_roads.csv",
    RUN_DRIVE_DIR / "hcmc_road_flood_risk_top10pct.geojson",
):
    if candidate.is_file():
        model_artifact.add_file(str(candidate), name=candidate.name)

wb_run.log_artifact(model_artifact)
run_url = wb_run.get_url()
wandb.finish()
print(f"\nĐã đồng bộ toàn bộ kết quả lên Weights & Biases thành công!")
if run_url:
    print(f"Xem Dashboard trực tuyến tại: {run_url}")